# Genizah text27b — Qwen3.8-27B text fine-tune: t27a (translation pilot) and t27b (joint run)

Answer-only SFT on human targets (plan 2026-10-21 section 4, amendment A1; non-thinking): **Qwen3.8-27B** from
`unsloth/Qwen3.8-27B-unsloth-bnb-4bit`, a **language-only LoRA** (r16, alpha 16, dropout 0) on the Gated DeltaNet `in_proj_qkv` / `in_proj_z` /
`out_proj`, the attention `q/k/v/o` and the MLP `gate/up/down` projections — never the vision tower, the MTP head or the
DeltaNet `in_proj_a` / `in_proj_b`. Every row is rendered with **`enable_thinking=False`** (no "Reasoning effort" sentence,
one empty think block, masked with the prompt) and the loss is on the answer only. At most 6,144 tokens a row
(nothing is truncated: a longer row fails G5), 16 rows a step, LR 1e-4 cosine with 3% warm-up, weight decay
0.01, adamw_8bit, seed 3407, 50-step saves, dev-monitor eval every 100 steps.

Kept from v2.5a: the session helpers (`train_session.py`: schedule lock, fixed row order, resume gate, time budget), the
Hub checkpoint helpers, the Hub-confirmed milestone pushes, `make_sft_config` / `make_trainer`, the prepared-dataloader
gate. The row order is the mixture file's own (materialised and seeded when the mixture was built), trained once.

| | t27a | t27b |
|---|---|---|
| Data (`isaacmg/genizah_text27b`, private) | `t27a/train.jsonl`: translate_en + translate_he, every condition, x2 | `t27b/train.jsonl`: the joint mixture (plan section 9.3) |
| Families | translate_en, translate_he | translate_en, translate_he, catalogue_json, describe_pgp, extract_relations, analyse_document, domain_reading, rag_context, canon_identify, canon_diff |
| Steps | train rows // 16 (plan: about 384) | train rows // 16 (plan: about 1,227) |
| Run / repositories | `genizah_t27a`; `isaacmg/qwen3.8-27b-genizah-t27a-ckpt`, `isaacmg/qwen3.8-27b-genizah-t27a-merged` (new, private) | `genizah_t27b`; `isaacmg/qwen3.8-27b-genizah-t27b-ckpt`, `isaacmg/qwen3.8-27b-genizah-t27b-merged` (new, private) |
| Hub-confirmed milestones | every 100 steps and the last | every 250 steps, 50 % and the last |
| Lambda safety stop | 13.0 h | 32.5 h |

**Knobs** (cell 1, or the environment of a headless run): `T27_ARM` (t27a | t27b), `T27_STAGE` (rung1 | rung2 | train;
default rung1), `T27_BATCH` (per-device batch 1, 2 or 4; accumulation makes 16 rows), `T27_SCRATCH_TAG` (names a rung's
scratch repositories; default: the launch time).

**The ladder — every stage is its own GO:**

1. `rung1` — smoke on `Qwen/Qwen3.5-2B` (the same qwen3_5 architecture): the pins resolve, G1–G8 pass, 15 steps with a
   kill at step 10 and an exact resume from the Hub's copy, scratch push, merge and reload test (G9 code path, G10).
2. `rung2` — the 27B: every gate, 30 timed steps (the worst-case batch first), the batch-geometry sweep, peak VRAM,
   step-0 dev loss per family, kill-and-resume at step 10, merge test (local, not pushed). Its GO estimate
   (measured s/step x steps x $1.99/h) prices the run.
3. `train` — the run: t27a first; t27b after t27a's evaluation.

| Gate | Cell | What must hold |
|---|---|---|
| G1 | 1 | the kernel satisfies `host_lambda.PIN_SETS["text27b"]`; fla and causal_conv1d import |
| G2 | 2 | pinned revision + mixture fingerprint; v0_2 schema; train split only; the arm's families; dev monitor disjoint from train, no reader-exposed reads |
| G3 | 3 | 0 LoRA tensors on vision, MTP, DeltaNet in_proj_a / in_proj_b; lora_A per group = the layer plan (400 on the 27B) |
| G4 | 3 | the DeltaNet layers run the fla / causal-conv1d kernels, not the torch fallback |
| G5 | 4 | render contract on every row; 0 rows over 6,144 tokens; answer supervised, prompt + empty think block masked |
| G6 | 5 | a non-zero gradient on every language lora_B, per group (DeltaNet / attention / MLP), none elsewhere |
| G7 | 8 | 32 rows per family decoded from the PREPARED train dataloader equal the cell-4 tokens and masks |
| G8 | 8 | prepared-vs-direct loss within 5%; the warm-up survived `make_sft_config` |
| G9 | 6, 8 | rungs: timed steps, geometry sweep, peak VRAM, GO estimate; every fresh start: step-0 dev loss per family (the 2x alarm reference) |
| G10 | 8, 9 | rungs: kill at step 10, exact resume from the Hub copy, end-of-run save, merge test (rung 1: reload + loss, scratch push) |

**The pins are exact.** `host_lambda.PIN_SETS["text27b"]` holds the package specifications (unsloth ==2026.10.7, unsloth_zoo ==2026.10.7, transformers ==5.17.0, trl ==1.15.0, peft ==0.21.2, accelerate ==1.15.0, bitsandbytes ==0.50.2, flash-linear-attention ==0.5.2, causal-conv1d ==1.7.0).
G1 asserts the specifications at runtime. Rung 1 resolves placeholder ranges; its versions are recorded as exact pins
and this notebook regenerated before the run.

**Data pins.** The data cell refuses to run until the arm's dataset revision (40-hex) and mixture fingerprint (64-hex) are
pinned: push the local mixture, then `derive_t27.py --t27a-revision <sha> --t27a-mixture <the pushed file>`
regenerates both notebooks.

**W&B:** project `genizah-text27b`, created as a private project before the first run (its visibility is set in W&B, not here);
never `qwen-hebrew-finetune`.

**To resume after a session ends: run every cell again, unedited, with the same `T27_ARM` / `T27_STAGE`.** The notebook
finds the newest checkpoint on the Hub, checks the stored schedule and row order, and continues with the rows not yet
trained. The export cell merges and pushes only a finished run.


In [ ]:
# Cell 1 — installs + env + run knobs + G1 (pin gate)
# The text27b pin set (host_lambda.PIN_SETS["text27b"]) controls the package versions.
# G1 asserts its specifications at runtime; resolved smoke versions become exact pins before training.
import os
import time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# Optional regional compilation stalls on symbolic shapes in this torch/Unsloth stack.
# The FLA/causal-conv1d kernels remain enabled and mandatory (G4); G9 measures this configuration.
os.environ["UNSLOTH_BLOCK_COMPILE"] = "0"
WANDB_PROJECT = 'genizah-text27b'   # private project; never the public 'qwen-hebrew-finetune' (its watcher follows the newest run)
assert WANDB_PROJECT != 'qwen-hebrew-finetune', "text27b runs must not log to the public project"
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# ---- run knobs (edit here, or set them in the environment of a headless run) ----
ARM = os.environ.get("T27_ARM", "t27a")          # t27a = translation pilot, t27b = joint run
STAGE = os.environ.get("T27_STAGE", "rung1")     # rung1 = 2B smoke, rung2 = 27B gates + timing, train = the run (each its own GO)
assert ARM in ('t27a', 't27b'), f"T27_ARM must be one of ('t27a', 't27b'), got {ARM!r}"
assert STAGE in ('rung1', 'rung2', 'train'), f"T27_STAGE must be one of ('rung1', 'rung2', 'train'), got {STAGE!r}"
SMOKE = STAGE == "rung1"                         # the smoke flag: the base becomes Qwen/Qwen3.5-2B (same qwen3_5 architecture)
SCRATCH_TAG = os.environ.get("T27_SCRATCH_TAG") or time.strftime("%m%d%H%M", time.gmtime())   # names a rung's scratch repos
ROWS_PER_STEP = 16                           # rows per optimizer step
PER_DEVICE_BATCH = int(os.environ.get("T27_BATCH", "1"))   # rung 2's geometry sweep may pick 2 or 4
assert PER_DEVICE_BATCH > 0 and ROWS_PER_STEP % PER_DEVICE_BATCH == 0, f"T27_BATCH must divide {ROWS_PER_STEP}"
GRAD_ACCUM = ROWS_PER_STEP // PER_DEVICE_BATCH
LADDER_STEPS, KILL_AT = 15, 10         # a rung: killed at step 10, resumed from the Hub to step 15 (G10)
print(f"arm {ARM}, stage {STAGE}, batch {PER_DEVICE_BATCH} x {GRAD_ACCUM}" + (f", scratch tag {SCRATCH_TAG}" if STAGE != "train" else ""))
%pip install -q "unsloth[colab-new]==2026.10.7" "unsloth_zoo==2026.10.7" "transformers==5.17.0" "trl==1.15.0" "peft==0.21.2" "accelerate==1.15.0" "bitsandbytes==0.50.2" "flash-linear-attention==0.5.2" "datasets==4.8.5" "tokenizers==0.23.3" "safetensors==0.8.0" "huggingface_hub==1.33.0" "hf_transfer==0.1.9" "wandb==0.31.0"
%pip install -q --no-build-isolation "causal-conv1d==1.7.0"   # builds against the installed torch
# ---- G1: the installed stack satisfies the text27b pin set ----
from importlib.metadata import version
from importlib.util import find_spec
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple
T27_PIN_STATUS = 'exact'
T27_KERNEL_PINS = {'unsloth': '==2026.10.7', 'unsloth_zoo': '==2026.10.7', 'transformers': '==5.17.0', 'trl': '==1.15.0', 'peft': '==0.21.2', 'accelerate': '==1.15.0', 'bitsandbytes': '==0.50.2', 'flash-linear-attention': '==0.5.2', 'causal-conv1d': '==1.7.0'}

def pin_problems(specifiers: Dict[str, str], get_version: Callable[[str], str]) -> List[str]:
    """Distributions that are missing or outside their PEP 440 specifier.

    :param specifiers: Distribution name -> specifier ('' = any version).
    :param get_version: ``importlib.metadata.version`` (or a stand-in); raises ``ImportError`` for a missing distribution.
    :returns: One line per problem; empty when the stack satisfies every specifier.
    """
    from packaging.specifiers import SpecifierSet
    problems: List[str] = []
    for name, spec in specifiers.items():
        try:
            have = get_version(name)
        except ImportError:   # importlib.metadata.PackageNotFoundError is an ImportError: a missing distribution is reported
            problems.append(f"{name}: not installed")
            continue
        if spec and not SpecifierSet(spec).contains(have, prereleases=True):
            problems.append(f"{name}: {have} does not satisfy {spec!r}")
    return problems

_problems = pin_problems(T27_KERNEL_PINS, version)
assert not _problems, "G1 pin gate FAILED (this kernel is not the text27b stack):\n  - " + "\n  - ".join(_problems)
_missing = [_m for _m in ("fla", "causal_conv1d") if find_spec(_m) is None]
assert not _missing, f"G1: the Gated DeltaNet fast-path kernels are not importable: {_missing}"
RESOLVED_PINS = {_n: version(_n) for _n in T27_KERNEL_PINS}
print(f"G1 OK ({T27_PIN_STATUS} pin set): {RESOLVED_PINS}")
if T27_PIN_STATUS == "placeholder":
    print("   PLACEHOLDER ranges: write these resolved versions into host_lambda.PIN_SETS['text27b'] as exact pins, "
          "then regenerate this notebook with derive_t27.py")

import torch
assert torch.cuda.is_available(), "No GPU — switch runtime to A100"


In [ ]:
# Cell 2 — data: the pinned text27b mixture (chat rows, MAP-STYLE, materialised order — no streaming) + G2 (data gate)
import json
from collections import Counter
from pathlib import Path
from google.colab import userdata
from huggingface_hub import login, snapshot_download

login(token=userdata.get("HF_TOKEN"))

DATA_REPO = 'isaacmg/genizah_text27b'   # private: the local v0_2 mixtures (<arm>/train.jsonl + mixture_manifest.json) and the dev monitor
# revision of DATA_REPO per arm, and the text fingerprint of <arm>/train.jsonl in file order (derive_t27.text_fingerprint)
DATA_REVISIONS = {'t27a': 'bf5217adcf96321c750ceca00b619d31cb7cb24d', 't27b': '3afbe0719fe11fba1f0e1bd9631b20778b299863'}
MIXTURE_FINGERPRINTS = {'t27a': '752654e970749b46cbd4f42c42c81d167bd8dfc6a8851c5f92781cbc93b2dc61', 't27b': 'd38aca30b2a7980c9afb4c73b4707537aa373ee6c1b3b4d6d9eccf45715ba8f2'}
ARM_FAMILIES = {'t27a': ('translate_en', 'translate_he'), 't27b': ('translate_en', 'translate_he', 'catalogue_json', 'describe_pgp', 'extract_relations', 'analyse_document', 'domain_reading', 'rag_context', 'canon_identify', 'canon_diff')}
ARM_REQUIRED = {'t27a': ('translate_en', 'translate_he'), 't27b': ('translate_en', 'translate_he', 'catalogue_json', 'describe_pgp', 'extract_relations')}
FAMILIES = ('translate_en', 'translate_he', 'catalogue_json', 'describe_pgp', 'extract_relations', 'analyse_document', 'domain_reading', 'rag_context', 'canon_identify', 'canon_diff')
CONDITIONS = ('edition', 'noised_edition', 'read', 'read_hard', 'read_partial', 'edition_rescript', 'literature')
BUILD = 'v0_2'
DEV_FILE = 'dev/dev_monitor.jsonl'   # E3 dev monitor: all dev translate + 40 catalogue + 20 relations rows, reader-exposed reads removed
TRAIN_FILE, MANIFEST_FILE = f"{ARM}/train.jsonl", f"{ARM}/mixture_manifest.json"
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

def check_pin(value: str, length: int, what: str, fix: str) -> str:
    """Refuse anything but a lowercase hex digest of ``length`` characters.

    :param value: The pinned value (a revision or a fingerprint).
    :param length: Expected number of hex characters (40 for a revision, 64 for a sha256).
    :param what: Name of the value, for the message.
    :param fix: What to do about it, for the message.
    :returns: ``value``.
    :raises RuntimeError: When ``value`` is a placeholder or malformed.
    """
    if len(value) != length or any(c not in "0123456789abcdef" for c in value):
        raise RuntimeError(f"{what} {value!r} is not a pinned {length}-hex value: {fix}")
    return value


def read_jsonl(path: Any) -> List[Dict[str, Any]]:
    """Rows of a JSONL file (blank lines skipped).

    :param path: File path.
    :returns: Parsed rows in file order.
    """
    import json
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


def text_fingerprint(rows: Iterable[Dict[str, Any]], keys: Sequence[str] = ("system", "user", "assistant")) -> str:
    """Order-sensitive sha256 over the message texts of rows (the digest ``text27b.rows.fingerprint`` computes).

    :param rows: Rows in their materialised order.
    :param keys: Message roles hashed per row.
    :returns: Hex digest.
    """
    import hashlib
    import json
    h = hashlib.sha256()
    for row in rows:
        for k in keys:
            value = next((m.get("content") for m in row.get("messages") or [] if m.get("role") == k), None)
            text = "" if value is None else (value if isinstance(value, str) else
                                             json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False))
            h.update(k.encode("utf-8"))
            h.update(b"\x1f")
            h.update(text.encode("utf-8"))
            h.update(b"\x1e")
        h.update(b"\x1d")
    return h.hexdigest()


def row_problems(row: Dict[str, Any], split: str, families: Sequence[str], conditions: Sequence[str],
                 build: str = "v0_2") -> List[str]:
    """Schema problems of one chat row (G2).

    :param row: ``{"messages": [system, user, assistant], "meta": {...}}``.
    :param split: The split every row of this file must carry.
    :param families: Families the row may belong to.
    :param conditions: Allowed input conditions.
    :param build: Required ``meta.build``.
    :returns: One line per problem; empty for a valid row.
    """
    problems: List[str] = []
    msgs = row.get("messages")
    roles = [m.get("role") for m in msgs] if isinstance(msgs, list) else None
    if roles != ["system", "user", "assistant"]:
        problems.append(f"roles {roles}, expected system / user / assistant")
    elif any(not isinstance(m.get("content"), str) for m in msgs):
        problems.append("a message content is not a string")
    elif not msgs[-1]["content"].strip():
        problems.append("empty assistant target")
    meta = row.get("meta") or {}
    if meta.get("split") != split:
        problems.append(f"split {meta.get('split')!r}, expected {split!r}")
    if meta.get("build") != build:
        problems.append(f"build {meta.get('build')!r}, expected {build!r}")
    if meta.get("family") not in families:
        problems.append(f"family {meta.get('family')!r} is not one of {list(families)}")
    if meta.get("condition") not in conditions:
        problems.append(f"condition {meta.get('condition')!r} is not one of {list(conditions)}")
    if meta.get("held_out_vision_benchmark") is True:
        problems.append("a vision-benchmark hold-out")
    if not meta.get("row_id"):
        problems.append("no row_id")
    return problems

# never train on an unpinned or unverified mixture: derive_t27.py writes both pins after the push
DATA_REVISION = check_pin(DATA_REVISIONS[ARM], 40, f"{ARM} dataset revision",
                          f"push the mixture, then regenerate this notebook with derive_t27.py --{ARM}-revision <sha> --{ARM}-mixture <file>")
MIXTURE_FINGERPRINT = check_pin(MIXTURE_FINGERPRINTS[ARM], 64, f"{ARM} mixture fingerprint",
                                f"regenerate this notebook with derive_t27.py --{ARM}-mixture <the pushed file>")
LOCAL = "/content/genizah_text27b"
DATA_DIR = Path(snapshot_download(DATA_REPO, repo_type="dataset", revision=DATA_REVISION, local_dir=LOCAL,
                                  allow_patterns=[TRAIN_FILE, MANIFEST_FILE, DEV_FILE, "README.md"]))
train_rows = read_jsonl(DATA_DIR / TRAIN_FILE)
dev_rows = read_jsonl(DATA_DIR / DEV_FILE)
MIX = json.loads((DATA_DIR / MANIFEST_FILE).read_text(encoding="utf-8"))
print(f"{ARM}: {len(train_rows):,} train rows, {len(dev_rows):,} dev-monitor rows; mixture manifest keys {sorted(MIX)}")

# ---- G2: the pinned mixture (fingerprint), train split only, this arm's families, the v0_2 schema; a clean dev monitor ----
_fp = text_fingerprint(train_rows)
assert _fp == MIXTURE_FINGERPRINT, f"G2: mixture fingerprint {_fp[:16]}… != pinned {MIXTURE_FINGERPRINT[:16]}… — not the pushed file"
_bad = [(_i, _p) for _i, _r in enumerate(train_rows) for _p in row_problems(_r, "train", ARM_FAMILIES[ARM], CONDITIONS, BUILD)]
assert not _bad, f"G2: {len(_bad)} train-row problems, e.g. {_bad[:5]}"
_bad = [(_i, _p) for _i, _r in enumerate(dev_rows) for _p in row_problems(_r, "dev", FAMILIES, CONDITIONS, BUILD)]
assert not _bad, f"G2: {len(_bad)} dev-monitor problems, e.g. {_bad[:5]}"
assert not [_r for _r in dev_rows if _r["meta"].get("reader_exposed") is True], "G2: reader-exposed reads in the dev monitor"
_train_docs = {str(_r["meta"]["pgpid"]) for _r in train_rows if _r["meta"].get("pgpid") not in (None, "")}
_shared = _train_docs & {str(_r["meta"]["pgpid"]) for _r in dev_rows if _r["meta"].get("pgpid") not in (None, "")}
assert not _shared, f"G2: {len(_shared)} documents in both train and the dev monitor, e.g. {sorted(_shared)[:5]}"
_shared = {_r["meta"]["row_id"] for _r in train_rows} & {_r["meta"]["row_id"] for _r in dev_rows}
assert not _shared, f"G2: {len(_shared)} row ids in both train and the dev monitor"
_fam = Counter(_r["meta"]["family"] for _r in train_rows)
assert set(ARM_REQUIRED[ARM]) <= set(_fam), f"G2: {ARM} needs {ARM_REQUIRED[ARM]}, the mixture has {sorted(_fam)}"
assert len(train_rows) >= ROWS_PER_STEP * LADDER_STEPS, "G2: fewer rows than a ladder rung needs"
_cond = Counter((_r["meta"]["family"], _r["meta"]["condition"]) for _r in train_rows)
print("G2 OK: train rows by family", dict(_fam))
print("   by family x condition", dict(sorted(_cond.items())))
print("   by language", dict(Counter(_r["meta"].get("lang_class") for _r in train_rows)))
print("   dev monitor by family", dict(Counter(_r["meta"]["family"] for _r in dev_rows)))


In [ ]:
# Cell 3 — model: Qwen3.8-27B (Unsloth bnb 4-bit; Qwen3.5-2B on rung 1) + language-only LoRA + G3 (adapter gate) + G4 (kernel fast path)
import inspect as _inspect
import re
from unsloth import FastModel

BASE_MODEL = 'Qwen/Qwen3.5-2B' if SMOKE else 'unsloth/Qwen3.8-27B-unsloth-bnb-4bit'
MAX_SEQ = 6144
LORA_RANK, LORA_ALPHA = 16, 16
# language layers only: DeltaNet in_proj_qkv / in_proj_z / out_proj, attention q / k / v / o, MLP gate / up / down; never the
# vision tower, the MTP head or DeltaNet in_proj_a / in_proj_b (PEFT full-matches this regex on every module name)
LORA_REGEX = r'(?!.*(?:visual|vision|mtp)).*\.layers\.\d+\.(?:linear_attn\.(?:in_proj_qkv|in_proj_z|out_proj)|self_attn\.(?:q_proj|k_proj|v_proj|o_proj)|mlp\.(?:gate_proj|up_proj|down_proj))'
EXPECTED_LORA_A = None if SMOKE else 400   # 27B: 48 DeltaNet x 3 + 16 attention x 4 + 64 MLP x 3 (rung 1: from the config)

model, tokenizer = FastModel.from_pretrained(BASE_MODEL, load_in_4bit=True, max_seq_length=MAX_SEQ,
                                             use_gradient_checkpointing="unsloth", full_finetuning=False)
model = FastModel.get_peft_model(
    model, target_modules=LORA_REGEX,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.0, bias="none", random_state=3407,
    use_gradient_checkpointing="unsloth",
)

def module_group(name: str) -> Optional[str]:
    """Module group of a parameter or module name.

    :param name: Dotted name (``...layers.3.linear_attn.in_proj_qkv.lora_A.default.weight``).
    :returns: ``vision``, ``mtp``, ``deltanet``, ``attention``, ``mlp``, or None outside those.
    """
    if "visual" in name or "vision" in name:
        return "vision"
    if "mtp" in name:
        return "mtp"
    if ".linear_attn." in name:
        return "deltanet"
    if ".self_attn." in name:
        return "attention"
    if ".mlp." in name:
        return "mlp"
    return None


def expected_lora_counts(layer_types: Sequence[str]) -> Dict[str, int]:
    """LoRA-adapted modules per group for a hybrid qwen3_5 stack with dense MLPs.

    DeltaNet layers adapt ``in_proj_qkv``, ``in_proj_z`` and ``out_proj`` (3), attention layers ``q/k/v/o`` (4), every
    layer's MLP ``gate/up/down`` (3).

    :param layer_types: ``config.layer_types`` (``linear_attention`` / ``full_attention`` per layer).
    :returns: ``{"deltanet": n, "attention": n, "mlp": n}``.
    :raises ValueError: On a layer type this rule does not know.
    """
    unknown = sorted(set(layer_types) - {"linear_attention", "full_attention"})
    if unknown:
        raise ValueError(f"unknown layer types {unknown}: the expected LoRA count covers linear_attention / full_attention")
    n_linear = sum(1 for t in layer_types if t == "linear_attention")
    n_full = sum(1 for t in layer_types if t == "full_attention")
    return {"deltanet": 3 * n_linear, "attention": 4 * n_full, "mlp": 3 * len(layer_types)}


def lora_problems(lora_a_names: Sequence[str], layer_types: Sequence[str],
                  expected_total: Optional[int] = None) -> Tuple[Dict[str, int], List[str]]:
    """G3: where the adapter sits, against the language-only plan.

    :param lora_a_names: Names of the ``lora_A`` parameters (or of the moved ``lora_B`` mapped to ``lora_A``).
    :param layer_types: ``config.layer_types`` of the language model.
    :param expected_total: Required total over the language groups (400 on the 27B), or None.
    :returns: ``(counts per group, problems)``.
    """
    import re
    counts = {"deltanet": 0, "attention": 0, "mlp": 0, "vision": 0, "mtp": 0, "other": 0}
    others: List[str] = []
    for n in lora_a_names:
        group = module_group(n)
        counts[group or "other"] += 1
        if group is None:
            others.append(n)
    problems: List[str] = []
    if counts["vision"]:
        problems.append(f"{counts['vision']} LoRA tensors on the vision tower (must be 0)")
    if counts["mtp"]:
        problems.append(f"{counts['mtp']} LoRA tensors on the MTP head (must be 0)")
    if others:
        problems.append(f"{len(others)} LoRA tensors outside the language groups, e.g. {others[:3]}")
    gates = [n for n in lora_a_names if re.search(r"\.linear_attn\.in_proj_[ab]\.", n)]
    if gates:
        problems.append(f"{len(gates)} LoRA tensors on DeltaNet in_proj_a / in_proj_b (must be 0), e.g. {gates[:2]}")
    expected = expected_lora_counts(layer_types)
    for group, n in expected.items():
        if counts[group] != n:
            problems.append(f"{group}: {counts[group]} lora_A tensors, expected {n}")
    total = sum(counts[g] for g in expected)
    if expected_total is not None and total != expected_total:
        problems.append(f"{total} language lora_A tensors, expected {expected_total}")
    return counts, problems


def repair_recurrent_kernel(modeling_module: Any, kernel_decorator: Callable[..., Any]) -> bool:
    """Rebind Unsloth's stale recurrent wrapper using Transformers' correct FLA kernel name.

    :param modeling_module: The loaded model's modeling module.
    :param kernel_decorator: Transformers' ``use_kernel_func_from_hub_with_fallback`` factory.
    :returns: Whether a known fallback wrapper was rebuilt; G4 still verifies the selected implementation.
    """
    import inspect
    wrapper = getattr(modeling_module, "torch_recurrent_gated_delta_rule", None)
    if not inspect.isfunction(wrapper) or not hasattr(wrapper, "__wrapped__"):
        return False
    if inspect.getclosurevars(wrapper).nonlocals.get("is_new_implementation") is not False:
        return False
    modeling_module.torch_recurrent_gated_delta_rule = kernel_decorator("fused_recurrent_gated_delta_rule", "fla")(wrapper.__wrapped__)
    return True


def fast_path_problems(layer: Any, module_flag: Optional[bool], modeling_module: Optional[Any] = None) -> List[str]:
    """G4: whether a Gated DeltaNet layer runs the fla / causal-conv1d kernels.

    :param layer: One ``linear_attn`` module (it holds its kernels as attributes, the torch fallbacks named ``torch_*``).
    :param module_flag: ``is_fast_path_available`` of its modeling module, or None when the module has none.
    :param modeling_module: Modeling module for Transformers versions with module-level kernel dispatch wrappers.
    :returns: One line per problem; empty when the fast path is in use.
    """
    kernels = ("causal_conv1d_fn", "causal_conv1d_update", "chunk_gated_delta_rule", "recurrent_gated_delta_rule")
    found = {k: getattr(layer, k) for k in kernels if hasattr(layer, k)}
    problems: List[str] = []
    if module_flag is False:
        problems.append("the modeling module reports is_fast_path_available = False (fla / causal-conv1d not importable there)")
    if not found and modeling_module is not None:
        import inspect
        wrappers = {"causal_conv1d_fn": "causal_conv1d", "causal_conv1d_update": "causal_conv1d",
                    "torch_chunk_gated_delta_rule": "fla", "torch_recurrent_gated_delta_rule": "fla"}
        for name, package in wrappers.items():
            fn = getattr(modeling_module, name, None)
            seen = set()
            verified = False
            while callable(fn) and id(fn) not in seen:
                seen.add(id(fn))
                captured = inspect.getclosurevars(fn).nonlocals if inspect.isfunction(fn) else {}
                if "is_new_implementation" in captured:
                    implementation = captured.get("implementation")
                    verified = (captured["is_new_implementation"] is True and callable(implementation)
                                and getattr(implementation, "__module__", "").split(".")[0] == package)
                    break
                fn = getattr(fn, "__wrapped__", None)
            if not verified:
                problems.append(f"{name}: cannot verify an active {package} kernel in the dispatch wrapper")
        return problems
    if not found and module_flag is None:
        problems.append(f"{type(layer).__name__} exposes neither the kernel attributes {kernels} nor is_fast_path_available: "
                        "check the fast path by hand before the run")
    for name, fn in found.items():
        if fn is None:
            problems.append(f"{name} is None (its kernel library is missing)")
        elif getattr(fn, "__name__", "").startswith("torch_"):
            problems.append(f"{name} is the torch fallback {fn.__name__}")
    return problems

# ---- G3: a language-only adapter — 0 vision / MTP / in_proj_a|b LoRA tensors, the planned lora_A count per module group ----
_text_cfg = getattr(model.config, "text_config", None) or model.config
LAYER_TYPES = list(_text_cfg.layer_types)
assert not getattr(_text_cfg, "num_experts", None), "G3: a MoE config — the expected count assumes dense MLPs"
_lora_a = [_n for _n, _ in model.named_parameters() if "lora_A" in _n]
LORA_COUNTS, _problems = lora_problems(_lora_a, LAYER_TYPES, EXPECTED_LORA_A)
if _problems:
    print("linear_attn modules found:", sorted({re.sub(r"\.\d+\.", ".N.", _n) for _n, _ in model.named_modules() if ".linear_attn." in _n})[:20])
assert not _problems, "G3 adapter gate FAILED:\n  - " + "\n  - ".join(_problems)
_trainable_other = [_n for _n, _p in model.named_parameters() if _p.requires_grad and "lora_" not in _n]
assert not _trainable_other, f"G3: non-LoRA parameters are trainable: {_trainable_other[:5]}"
print(f"G3 OK: {sum(LORA_COUNTS[_g] for _g in ('deltanet', 'attention', 'mlp'))} language lora_A tensors {LORA_COUNTS} on {len(LAYER_TYPES)} layers")

# ---- G4: the Gated DeltaNet layers run the fla / causal-conv1d kernels, not the torch fallback ----
REQUIRE_FAST_PATH = True
_lin = [_m for _n, _m in model.named_modules() if _n.endswith(".linear_attn")]
assert _lin, "G4: no linear_attn modules — not a qwen3_5 model?"
_modeling_module = _inspect.getmodule(type(_lin[0]))
from transformers.integrations.hub_kernels import use_kernel_func_from_hub_with_fallback
if repair_recurrent_kernel(_modeling_module, use_kernel_func_from_hub_with_fallback):
    print("Rebound the recurrent DeltaNet wrapper to fused_recurrent_gated_delta_rule; G4 verifies its implementation")
_problems = fast_path_problems(_lin[0], getattr(_modeling_module, "is_fast_path_available", None), _modeling_module)
if _problems and REQUIRE_FAST_PATH:
    raise AssertionError("G4 kernel fast-path gate FAILED:\n  - " + "\n  - ".join(_problems))
print("G4 OK: the DeltaNet layers run the fla / causal-conv1d kernels" if not _problems else f"⚠️ G4 fast path missing: {_problems}")
print(f"base {BASE_MODEL}; trainable {sum(_p.numel() for _p in model.parameters() if _p.requires_grad) / 1e6:.1f}M parameters")


In [ ]:
# Cell 4 — render (enable_thinking=False) + tokenize once + response-only labels + collator + G5 (render, length, mask)
import hashlib
import unicodedata
import numpy as np
from torch.utils.data import Dataset as _TorchDataset

_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer   # the text tokenizer under a processor
CHAT_TEMPLATE = getattr(_tok, "chat_template", None) or getattr(tokenizer, "chat_template", None)
assert CHAT_TEMPLATE and "enable_thinking" in CHAT_TEMPLATE, "G5: the chat template has no enable_thinking switch"
TEMPLATE_SHA256 = hashlib.sha256(CHAT_TEMPLATE.encode("utf-8")).hexdigest()
# sha256 of the local template rung 0 checked: informational (the render contract below is the gate)
LOCAL_TEMPLATE_SHA256 = 'c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041'
print(f"chat template sha256 {TEMPLATE_SHA256[:16]}…" + ("" if LOCAL_TEMPLATE_SHA256 is None else
      (" = the local copy's" if TEMPLATE_SHA256 == LOCAL_TEMPLATE_SHA256 else f" != the local copy's {LOCAL_TEMPLATE_SHA256[:16]}…")))
THINK_IDS = [_tok.convert_tokens_to_ids(_t) for _t in ("<think>", "</think>")]
PAD_ID = _tok.pad_token_id if _tok.pad_token_id is not None else _tok.eos_token_id

def nfc_messages(messages: Sequence[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Copies of chat turns with NFC-normalised string contents.

    The tokenizer normalises text to NFC, so the supervised tokens of a decomposed answer (``t`` + U+0323) decode
    to the composed form; rendering NFC text keeps the mask guardrail's answer comparison exact (fix pass F3).

    :param messages: Chat turns.
    :returns: New turn dicts; non-string contents are copied unchanged.
    """
    import unicodedata
    return [dict(m, content=unicodedata.normalize("NFC", m["content"])) if isinstance(m.get("content"), str) else dict(m)
            for m in messages]


def render_problems(rendered: str, messages: Sequence[Dict[str, Any]], empty_think: str = "<think>\n\n</think>\n\n",
                    effort_marker: str = "Reasoning effort", im_end: str = "<|im_end|>\n") -> List[str]:
    """G5: the ``enable_thinking=False`` render contract (the rule of ``text27b.rows.check_render``).

    :param rendered: The chat template's output for a training row.
    :param messages: The rendered messages (ending in the assistant target).
    :param empty_think: The empty think block.
    :param effort_marker: Start of the sentence the template injects when thinking is on.
    :param im_end: End-of-turn marker.
    :returns: One line per broken rule; empty when the render is right.
    """
    problems: List[str] = []
    if effort_marker in rendered:
        problems.append("the reasoning-effort sentence is in the render (enable_thinking was not honoured)")
    n_assistant = sum(1 for m in messages if m.get("role") == "assistant")
    if rendered.count(empty_think) != n_assistant or rendered.count("<think>") != n_assistant:
        problems.append(f"{rendered.count(empty_think)} empty think blocks and {rendered.count('<think>')} <think> tags, "
                        f"expected {n_assistant} of each")
    if not messages or messages[-1].get("role") != "assistant":
        problems.append("the last turn is not the assistant target")
    elif not rendered.endswith(empty_think + str(messages[-1].get("content") or "").strip() + im_end):
        problems.append("the render does not end with the empty think block + the answer + <|im_end|>")
    return problems


def split_rendered(rendered: str, content: str, im_end: str = "<|im_end|>\n",
                   assistant_open: str = "<|im_start|>assistant\n",
                   empty_think: str = "<think>\n\n</think>\n\n") -> Tuple[str, str]:
    """Split a rendered row into the masked prompt and the supervised completion.

    :param rendered: The chat template's output.
    :param content: The assistant target (the template strips it).
    :param im_end: End-of-turn marker.
    :param assistant_open: Assistant header.
    :param empty_think: The empty think block.
    :returns: ``(prompt, completion)``; the prompt ends with the assistant header and the empty think block, the
        completion is the stripped answer + ``<|im_end|>\\n``.
    :raises ValueError: When the render does not end with the completion or the prompt does not end with the header.
    """
    completion = str(content or "").strip() + im_end
    if not rendered.endswith(completion):
        raise ValueError("the render does not end with the answer + <|im_end|>")
    prompt = rendered[: len(rendered) - len(completion)]
    if not prompt.endswith(assistant_open + empty_think):
        raise ValueError("the prompt does not end with the assistant header and the empty think block")
    return prompt, completion


def prompt_token_count(offsets: Sequence[Sequence[int]], boundary: int) -> int:
    """Number of leading tokens that lie in the prompt (character offsets end at or before ``boundary``).

    :param offsets: ``(start, end)`` character offsets of every token, in order.
    :param boundary: Length of the prompt in characters.
    :returns: The prompt's token count; every later token starts at or after ``boundary``.
    :raises ValueError: When a token straddles the boundary or a prompt token follows an answer token.
    """
    n, in_answer = 0, False
    for start, end in offsets:
        if end <= boundary and not in_answer:
            n += 1
        elif start >= boundary:
            in_answer = True
        else:
            raise ValueError(f"token ({start}, {end}) straddles the prompt / answer boundary at {boundary}")
    return n


def supervised_problems(ids: Sequence[int], labels: Sequence[int], n_prompt: int, prompt: str, completion: str,
                        decode: Callable[[Sequence[int]], str], think_ids: Sequence[int] = (),
                        ignore_index: int = -100) -> List[str]:
    """G5 / G7 mask guardrail on one unpadded row: the answer is exactly the supervised part.

    :param ids: Input ids of the row.
    :param labels: Its labels.
    :param n_prompt: Prompt token count (the masked prefix).
    :param prompt: The prompt text (system + user + assistant header + empty think block).
    :param completion: The answer + ``<|im_end|>\\n``.
    :param decode: Token ids -> text, lossless (special tokens kept).
    :param think_ids: Ids of ``<think>`` / ``</think>``; none may be supervised.
    :param ignore_index: The masked label value.
    :returns: One line per problem; empty when the mask is right.
    """
    ids, labels = list(ids), list(labels)
    problems: List[str] = []
    if len(ids) != len(labels):
        problems.append(f"{len(ids)} ids but {len(labels)} labels")
        return problems
    masked = [i for i, label in enumerate(labels) if label == ignore_index]
    if masked != list(range(n_prompt)):
        problems.append(f"labels mask {len(masked)} tokens, not exactly the {n_prompt}-token prompt prefix")
    if labels[n_prompt:] != ids[n_prompt:]:
        problems.append("supervised labels differ from the input ids")
    if not ids[n_prompt:]:
        problems.append("nothing is supervised")
    if decode(ids[:n_prompt]) != prompt:
        problems.append("the masked prefix does not decode to the prompt (system + user + assistant header + empty think block)")
    if decode(ids[n_prompt:]) != completion:
        problems.append("the supervised tokens do not decode to the answer + <|im_end|>")
    if set(think_ids) & set(ids[n_prompt:]):
        problems.append("a think tag is supervised")
    return problems



def render(messages):
    """The training render of a row: the chat template with enable_thinking=False (no generation prompt), NFC text
    (the tokenizer normalises to NFC; pass nfc_messages(...) so the answer compares equal)."""
    return unicodedata.normalize("NFC", _tok.apply_chat_template(list(messages), chat_template=CHAT_TEMPLATE,
                                                                 tokenize=False, enable_thinking=False))


def decode(ids):
    """Lossless decode (special tokens kept, no clean-up)."""
    return _tok.decode(list(ids), skip_special_tokens=False, clean_up_tokenization_spaces=False)


class TokenizedRows(_TorchDataset):
    """Rows rendered and tokenized once (G5 render contract on every row); labels mask the prompt prefix."""

    def __init__(self, rows):
        self.rows, self.meta, self.ids, self.n_prompt = rows, [r["meta"] for r in rows], [], []
        for k, row in enumerate(rows):
            msgs = nfc_messages(row["messages"])
            text = render(msgs)
            problems = render_problems(text, msgs)
            assert not problems, f"G5 render contract FAILED on row {k} ({row['meta'].get('row_id')}): {problems}"
            prompt, _ = split_rendered(text, msgs[-1]["content"])
            enc = _tok(text, add_special_tokens=False, return_offsets_mapping=True)
            self.n_prompt.append(prompt_token_count(enc["offset_mapping"], len(prompt)))
            self.ids.append(np.asarray(enc["input_ids"], dtype=np.int32))

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        ids = torch.from_numpy(self.ids[i].astype(np.int64))
        labels = ids.clone()
        labels[: self.n_prompt[i]] = -100
        return {"input_ids": ids, "labels": labels}

    def column(self, name):
        return [m.get(name) for m in self.meta]

    def parts(self, i):
        """(prompt, completion) text of row i (NFC, as tokenized)."""
        msgs = nfc_messages(self.rows[i]["messages"])
        return split_rendered(render(msgs), msgs[-1]["content"])


def collate(items):
    """Right-padded batch: input_ids, attention_mask, labels (-100 on the prompt and the padding)."""
    n = max(len(it["input_ids"]) for it in items)
    ids = torch.full((len(items), n), PAD_ID, dtype=torch.long)
    labels = torch.full((len(items), n), -100, dtype=torch.long)
    mask = torch.zeros((len(items), n), dtype=torch.long)
    for k, it in enumerate(items):
        m = len(it["input_ids"])
        ids[k, :m], labels[k, :m], mask[k, :m] = it["input_ids"], it["labels"], 1
    return {"input_ids": ids, "attention_mask": mask, "labels": labels}


train_ds = TokenizedRows(train_rows)
dev_sets = {_f: TokenizedRows([_r for _r in dev_rows if _r["meta"]["family"] == _f])
            for _f in sorted({_r["meta"]["family"] for _r in dev_rows})}   # one eval set per family: eval_<family>_loss

# ---- G5: 0 rows over MAX_SEQ (nothing is truncated); the mask guardrail on 2 rows per family ----
_lens = np.array([len(_x) for _x in train_ds.ids])
_over = Counter(_m["family"] for _m, _n in zip(train_ds.meta, _lens) if _n > MAX_SEQ)
_over_dev = Counter(_f for _f, _d in dev_sets.items() for _x in _d.ids if len(_x) > MAX_SEQ)
assert not _over and not _over_dev, f"G5: rows over MAX_SEQ={MAX_SEQ}: train {dict(_over)}, dev {dict(_over_dev)} — fix the mixture"
_guard = {}
for _i, _m in enumerate(train_ds.meta):
    if len(_guard.setdefault(_m["family"], [])) < 2:
        _guard[_m["family"]].append(_i)
for _f, _idx in _guard.items():
    for _i in _idx:
        _item = train_ds[_i]
        _prompt, _completion = train_ds.parts(_i)
        _p = supervised_problems(_item["input_ids"].tolist(), _item["labels"].tolist(), train_ds.n_prompt[_i], _prompt, _completion,
                                 decode, THINK_IDS)
        assert not _p, f"G5 mask guardrail FAILED on {_f} row {_i}: {_p}"
_sup = int(sum(len(_x) - _n for _x, _n in zip(train_ds.ids, train_ds.n_prompt)))
print(f"G5 OK: {len(train_ds):,} rows rendered with enable_thinking=False (one empty think block each, no effort sentence); "
      f"longest {int(_lens.max()):,} tokens (<= {MAX_SEQ:,}); {int(_lens.sum()):,} tokens, {_sup:,} supervised; "
      f"mask guardrail on {sum(len(_v) for _v in _guard.values())} rows; dev monitor {sum(len(_d) for _d in dev_sets.values())} rows "
      f"in {len(dev_sets)} families")


In [ ]:
# Cell 5 — G6 gradient flow per module group (cheap; run BEFORE burning GPU-hours)

def grad_group_problems(entries: Iterable[Tuple[str, Optional[float]]],
                        expected: Dict[str, int]) -> Tuple[Dict[str, Tuple[int, float]], List[str]]:
    """G6: gradient norm per module group over the ``lora_B`` tensors after one backward pass.

    :param entries: ``(parameter name, gradient L2 norm or None)`` per ``lora_B`` tensor.
    :param expected: Tensors per language group that must carry a non-zero gradient (:func:`expected_lora_counts`).
    :returns: ``(group -> (tensors with a non-zero gradient, L2 norm of the group), problems)``.
    """
    import math
    n: Dict[str, int] = {}
    sq: Dict[str, float] = {}
    for name, norm in entries:
        group = module_group(name) or "other"
        if norm is None or not norm > 0:
            continue
        n[group] = n.get(group, 0) + 1
        sq[group] = sq.get(group, 0.0) + float(norm) ** 2
    summary = {g: (n[g], math.sqrt(sq[g])) for g in n}
    problems: List[str] = []
    for group in sorted(set(n) - set(expected)):
        problems.append(f"{n[group]} {group} lora_B tensors got a gradient (only the language groups may)")
    for group, want in expected.items():
        if n.get(group, 0) != want:
            problems.append(f"{group}: {n.get(group, 0)} of {want} lora_B tensors got a non-zero gradient")
    return summary, problems

FastModel.for_training(model)
_vb = {_k: _v.to(model.device) for _k, _v in collate([train_ds[0]]).items()}
model(**_vb).loss.backward()
_entries = [(_n, (_p.grad.detach().float().norm().item() if _p.grad is not None else None))
            for _n, _p in model.named_parameters() if "lora_B" in _n]
model.zero_grad(set_to_none=True)
GRAD_NORMS, _problems = grad_group_problems(_entries, expected_lora_counts(LAYER_TYPES))
assert not _problems, "G6 gradient gate FAILED:\n  - " + "\n  - ".join(_problems)
print("G6 OK: lora_B gradients " + ", ".join(f"{_g} {_n} tensors |g|={_v:.2e}" for _g, (_n, _v) in sorted(GRAD_NORMS.items())))
del _vb


In [ ]:
# Cell 6 — G9 throughput on the ladder rungs: timed steps (the worst-case batch first), batch-geometry sweep, peak VRAM, GO estimate
import statistics
RUN_BENCH = STAGE != "train"
TIMED_STEPS = 30 if STAGE == "rung2" else 5
BATCH_GEOMETRIES = [(1, 16), (2, 8), (4, 4)]   # (per-device batch, accumulation): 16 rows a step each
PRICE_PER_H = 1.99             # A100 40 GB on demand (plan section 4): the GO number = measured s/step x steps x price
MERGE_H = {'t27a': 1.5, 't27b': 2.0}               # merge + push hours after training, per arm

def pick_geometry(results: Dict[Tuple[int, int], Optional[float]]) -> Tuple[int, int]:
    """The fastest batch geometry that did not run out of memory.

    :param results: ``(per-device batch, accumulation) -> seconds per step``, None for OOM.
    :returns: The fastest geometry (ties: the smaller per-device batch).
    :raises RuntimeError: When every geometry ran out of memory.
    """
    ok = {g: t for g, t in results.items() if t is not None}
    if not ok:
        raise RuntimeError("every batch geometry ran out of memory")
    return min(ok, key=lambda g: (ok[g], g[0]))


def go_estimate(s_per_step: float, steps: int, price_per_h: float, extra_h: float) -> Dict[str, float]:
    """The GO number: hours and dollars of a run from a measured step time.

    :param s_per_step: Seconds per optimizer step.
    :param steps: Steps of the run.
    :param price_per_h: Instance price per hour.
    :param extra_h: Hours after training (merge, push).
    :returns: ``{"train_h", "total_h", "usd"}``.
    """
    train_h = s_per_step * steps / 3600
    total_h = train_h + extra_h
    return {"train_h": round(train_h, 2), "total_h": round(total_h, 2), "usd": round(total_h * price_per_h, 2)}



def timed_step(rows, per_device, accum):
    """Seconds of one optimizer step's forward + backward passes over ``rows`` (no optimizer update)."""
    torch.cuda.synchronize()
    t0 = time.time()
    for k in range(accum):
        b = {key: v.to(model.device) for key, v in collate([train_ds[i] for i in rows[k * per_device:(k + 1) * per_device]]).items()}
        (model(**b).loss / accum).backward()
    model.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    return time.time() - t0


if RUN_BENCH:
    FastModel.for_training(model)
    _lens = [len(_x) for _x in train_ds.ids]
    WORST = sorted(range(len(train_ds)), key=lambda _i: -_lens[_i])[:ROWS_PER_STEP]   # the longest rows: the worst-case step
    timed_step(WORST[:1], 1, 1)                                                          # warm-up: kernel autotune, allocator
    BENCH = {}
    for _bs, _ga in BATCH_GEOMETRIES:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        try:
            BENCH[(_bs, _ga)] = timed_step(WORST, _bs, _ga)
            print(f"batch {_bs}x{_ga}: worst-case step {BENCH[(_bs, _ga)]:.1f} s, peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
        except torch.cuda.OutOfMemoryError:   # an expected outcome of the sweep: that geometry does not fit
            BENCH[(_bs, _ga)] = None
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            print(f"batch {_bs}x{_ga}: OOM")
    BEST_GEOMETRY = pick_geometry(BENCH)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    _n = len(train_ds)
    _times = [timed_step(WORST if _s == 0 else [((_s - 1) * ROWS_PER_STEP + _j) % _n for _j in range(ROWS_PER_STEP)],
                         PER_DEVICE_BATCH, GRAD_ACCUM) for _s in range(TIMED_STEPS)]
    PEAK_GIB = torch.cuda.max_memory_allocated() / 2**30
    S_PER_STEP = statistics.median(_times[1:]) if len(_times) > 1 else _times[0]
    _steps = _n // ROWS_PER_STEP
    GO = go_estimate(S_PER_STEP, _steps, PRICE_PER_H, MERGE_H[ARM])
    print(f"G9: {TIMED_STEPS} timed steps at {PER_DEVICE_BATCH}x{GRAD_ACCUM}: worst-case {_times[0]:.1f} s, median {S_PER_STEP:.1f} s, "
          f"peak VRAM {PEAK_GIB:.1f} GiB; fastest geometry {BEST_GEOMETRY}"
          + ("" if BEST_GEOMETRY == (PER_DEVICE_BATCH, GRAD_ACCUM) else f" — set T27_BATCH={BEST_GEOMETRY[0]} for the run"))
    print(f"GO estimate for {ARM}: {_steps:,} steps x {S_PER_STEP:.1f} s = {GO['train_h']} h + {MERGE_H[ARM]} h merge/push "
          f"= {GO['total_h']} h, about ${GO['usd']} at ${PRICE_PER_H}/h")
else:
    print("G9 timing skipped on the run (rung 2 measured it; W&B times every step)")


In [ ]:

# ---- v22b session helpers (inlined from src/finetuning/qwen_hebrew/train_session.py; keep in sync) ----
import gc, glob, hashlib, json, math, os, time
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
import numpy as np
import torch
from torch.utils.data import Dataset as _TorchDataset, SequentialSampler
from transformers import TrainerCallback


REQUIRED_CHECKPOINT_FILES: Tuple[str, ...] = ("trainer_state.json", "optimizer.pt", "scheduler.pt")
WEIGHT_FILES: Tuple[str, ...] = ("adapter_model.safetensors", "adapter_model.bin", "model.safetensors", "pytorch_model.bin")
PLAN_IGNORED_KEYS: Tuple[str, ...] = ("order", "order_sha256", "created_at")


def materialize_order(n_rows: int, epochs: int, seed: int) -> List[int]:
    """Deterministic multi-epoch row order: one independent permutation per epoch, concatenated.

    :param n_rows: Dataset size.
    :type n_rows: int
    :param epochs: Number of passes to lay out (use the ceiling of ``max_steps × batch / n_rows``).
    :type epochs: int
    :param seed: Base seed; epoch ``e`` uses ``seed + e``.
    :type seed: int
    :return: ``n_rows × epochs`` dataset indices.
    :rtype: List[int]
    :raises ValueError: On non-positive sizes.
    """
    if n_rows <= 0 or epochs <= 0:
        raise ValueError("n_rows and epochs must be positive")
    order: List[int] = []
    for e in range(epochs):
        order.extend(np.random.default_rng(seed + e).permutation(n_rows).tolist())
    return order


def rows_consumed(global_step: int, per_device_batch: int, grad_accum: int, world_size: int = 1) -> int:
    """Rows the trainer has consumed after ``global_step`` optimizer steps.

    :param global_step: Restored ``trainer_state.global_step``.
    :type global_step: int
    :param per_device_batch: ``per_device_train_batch_size``.
    :type per_device_batch: int
    :param grad_accum: ``gradient_accumulation_steps``.
    :type grad_accum: int
    :param world_size: Number of processes (1 on Colab).
    :type world_size: int
    :return: Number of dataset rows already seen.
    :rtype: int
    """
    return global_step * per_device_batch * grad_accum * world_size


def remaining_rows(order: Sequence[int], global_step: int, per_device_batch: int, grad_accum: int,
                   world_size: int = 1) -> List[int]:
    """The tail of the materialised order a resumed session still has to train on.

    :param order: Output of :func:`materialize_order`.
    :type order: Sequence[int]
    :param global_step: Restored ``global_step`` (0 for a fresh run).
    :type global_step: int
    :param per_device_batch: ``per_device_train_batch_size``.
    :type per_device_batch: int
    :param grad_accum: ``gradient_accumulation_steps``.
    :type grad_accum: int
    :param world_size: Number of processes.
    :type world_size: int
    :return: Remaining indices (possibly empty when the order is exhausted).
    :rtype: List[int]
    """
    done = rows_consumed(global_step, per_device_batch, grad_accum, world_size)
    return list(order[done:])


def epochs_for(max_steps: int, n_rows: int, per_device_batch: int, grad_accum: int, world_size: int = 1) -> int:
    """Smallest number of passes whose row count covers ``max_steps`` optimizer steps.

    :param max_steps: Planned optimizer steps.
    :type max_steps: int
    :param n_rows: Dataset size.
    :type n_rows: int
    :param per_device_batch: ``per_device_train_batch_size``.
    :type per_device_batch: int
    :param grad_accum: ``gradient_accumulation_steps``.
    :type grad_accum: int
    :param world_size: Number of processes.
    :type world_size: int
    :return: Passes (at least 1).
    :rtype: int
    """
    need = rows_consumed(max_steps, per_device_batch, grad_accum, world_size)
    return max(1, -(-need // n_rows))


@dataclass
class BudgetState:
    """What the time-budget callback decided, for logging and tests.

    :param stopped: Whether training was ended by the budget.
    :param elapsed_s: Elapsed wall-clock seconds when the decision was taken.
    :param step: ``global_step`` at that point.
    """
    stopped: bool = False
    elapsed_s: float = 0.0
    step: int = 0


class TimeBudgetCallback:
    """Stop training cleanly (save + push) when a wall-clock budget is spent.

    Implemented against the ``transformers.TrainerCallback`` interface without importing
    transformers at module import time, so the pure logic is unit-testable on a CPU box.
    Attach with ``trainer.add_callback(TimeBudgetCallback(budget_s=21.5 * 3600))``.

    :param budget_s: Wall-clock budget in seconds measured from the first :meth:`on_train_begin`
        (from callback creation until then).
    :type budget_s: float
    :param min_steps_between_saves: Do not stop within this many steps of the last regular
        save unless the budget is exhausted anyway (avoids a redundant checkpoint).
    :type min_steps_between_saves: int
    :param clock: Time source (injectable for tests).
    """

    def __init__(self, budget_s: float, min_steps_between_saves: int = 0, clock=time.monotonic) -> None:
        self.budget_s = float(budget_s)
        self.min_steps_between_saves = int(min_steps_between_saves)
        self._clock = clock
        self._t0 = clock()
        self._started = False
        self.state = BudgetState()

    def on_train_begin(self, args=None, state=None, control=None, **kwargs):
        """Start the clock when training first starts.

        Only the first call resets it: an in-session retry calls ``train()`` again, and the
        budget has to keep counting from the session's first step or the retry would run past
        Colab's ceiling.

        :return: ``control`` unchanged.
        """
        if not self._started:
            self._t0 = self._clock()
            self._started = True
        return control

    def elapsed(self) -> float:
        """Seconds since the clock was (re)started.

        :return: Elapsed seconds.
        :rtype: float
        """
        return self._clock() - self._t0

    def should_stop(self, global_step: int) -> bool:
        """Pure decision: budget spent?

        :param global_step: Current optimizer step.
        :type global_step: int
        :return: True when training should end now.
        :rtype: bool
        """
        return self.elapsed() >= self.budget_s

    def on_step_end(self, args=None, state=None, control=None, **kwargs):
        """Request save + stop at the end of the step that crosses the budget.

        :return: ``control`` with ``should_save`` and ``should_training_stop`` set when stopping.
        """
        step = int(getattr(state, "global_step", 0) or 0)
        if not self.state.stopped and self.should_stop(step):
            self.state = BudgetState(stopped=True, elapsed_s=self.elapsed(), step=step)
            if control is not None:
                control.should_save = True
                control.should_training_stop = True
            print(f"[TimeBudgetCallback] budget {self.budget_s/3600:.1f} h spent at step {step}; saving and stopping", flush=True)
        return control


def describe_session(n_rows: int, max_steps: int, per_device_batch: int, grad_accum: int, global_step: int,
                     seed: int, world_size: int = 1) -> str:
    """Human-readable summary of what a session will train on.

    :param n_rows: Dataset size.
    :type n_rows: int
    :param max_steps: Planned optimizer steps.
    :type max_steps: int
    :param per_device_batch: ``per_device_train_batch_size``.
    :type per_device_batch: int
    :param grad_accum: ``gradient_accumulation_steps``.
    :type grad_accum: int
    :param global_step: Restored step (0 = fresh).
    :type global_step: int
    :param seed: Order seed.
    :type seed: int
    :param world_size: Processes.
    :type world_size: int
    :return: One-line description.
    :rtype: str
    """
    ep = epochs_for(max_steps, n_rows, per_device_batch, grad_accum, world_size)
    order = materialize_order(n_rows, ep, seed)
    rem = remaining_rows(order, global_step, per_device_batch, grad_accum, world_size)
    steps_left = len(rem) // (per_device_batch * grad_accum * world_size)
    return (f"{n_rows} rows × {ep} pass(es) = {len(order)} ordered rows; resumed at step {global_step} → "
            f"{len(rem)} rows / {steps_left} steps remain (target max_steps {max_steps}); seed {seed}")


def warmup_steps_for(max_steps: int, warmup_ratio: float) -> int:
    """Warm-up steps the trainer derives from a ratio (``TrainingArguments.get_warmup_steps``).

    :param max_steps: Planned optimizer steps.
    :type max_steps: int
    :param warmup_ratio: ``warmup_ratio`` training argument.
    :type warmup_ratio: float
    :return: ``ceil(max_steps × warmup_ratio)``.
    :rtype: int
    """
    return math.ceil(max_steps * warmup_ratio)


def cosine_warmup_lr(step: int, base_lr: float, warmup_steps: int, total_steps: int) -> float:
    """Learning rate after ``step`` scheduler steps under linear warm-up + half-cosine decay.

    Mirrors ``transformers.get_cosine_schedule_with_warmup`` (``num_cycles=0.5``): this is the
    rate the optimizer uses for optimizer step ``step + 1``, and what a correctly restored
    scheduler reports right after ``resume_from_checkpoint`` at ``global_step == step``.

    :param step: Scheduler steps taken so far (the restored ``global_step``).
    :type step: int
    :param base_lr: ``learning_rate`` training argument.
    :type base_lr: float
    :param warmup_steps: Warm-up length in steps.
    :type warmup_steps: int
    :param total_steps: ``max_steps``.
    :type total_steps: int
    :return: Scheduled learning rate.
    :rtype: float
    """
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))


def order_fingerprint(order: Sequence[int]) -> str:
    """SHA-256 of a row order (detects a changed permutation between sessions).

    :param order: Dataset indices in training order.
    :type order: Sequence[int]
    :return: Hex digest.
    :rtype: str
    """
    return hashlib.sha256(",".join(str(int(i)) for i in order).encode("ascii")).hexdigest()


def rows_fingerprint(columns: Sequence[Sequence[object]]) -> str:
    """SHA-256 over the rows of a dataset, given some of its columns (index → row mapping check).

    :param columns: Equal-length column lists (for example image hash, source, task, question, answer).
    :type columns: Sequence[Sequence[object]]
    :return: Hex digest.
    :rtype: str
    :raises ValueError: When the columns differ in length.
    """
    lengths = {len(c) for c in columns}
    if len(lengths) > 1:
        raise ValueError(f"columns differ in length: {sorted(lengths)}")
    digest = hashlib.sha256()
    for row in zip(*columns):
        digest.update(("\x1f".join("" if v is None else str(v) for v in row) + "\x1e").encode("utf-8"))
    return digest.hexdigest()


def build_session_plan(order: Sequence[int], **schedule: object) -> Dict[str, object]:
    """Everything that defines the training schedule, as one JSON-serialisable record.

    :param order: The materialised row order the run trains on.
    :type order: Sequence[int]
    :param schedule: Schedule-defining settings (steps, batch, accumulation, learning rate,
        warm-up, scheduler, optimiser, seeds, dataset repo and revision, row fingerprint, ...).
    :type schedule: object
    :return: The settings plus ``order_len`` and ``order_sha256``.
    :rtype: Dict[str, object]
    """
    plan: Dict[str, object] = dict(schedule)
    plan["order_len"] = len(order)
    plan["order_sha256"] = order_fingerprint(order)
    return plan


def diff_session_plan(stored: Dict[str, object], current: Dict[str, object],
                      ignore: Iterable[str] = PLAN_IGNORED_KEYS) -> List[str]:
    """Differences between the plan stored at the first launch and this session's plan.

    :param stored: Plan read back from the checkpoint repository.
    :type stored: Dict[str, object]
    :param current: Plan built from this session's settings.
    :type current: Dict[str, object]
    :param ignore: Keys that do not define the schedule (the order itself is taken from the stored
        plan, so its fingerprint is not compared here).
    :type ignore: Iterable[str]
    :return: One line per differing key; empty when the schedules are the same.
    :rtype: List[str]
    """
    skip = set(ignore)
    missing = object()
    problems: List[str] = []
    for key in sorted((set(stored) | set(current)) - skip):
        a, b = stored.get(key, missing), current.get(key, missing)
        if a is missing:
            problems.append(f"{key}: not in the stored plan, now {b!r}")
        elif b is missing:
            problems.append(f"{key}: stored {a!r}, missing now")
        elif a != b:
            problems.append(f"{key}: stored {a!r}, now {b!r}")
    return problems


def checkpoint_step(path: str) -> Optional[int]:
    """``global_step`` of a checkpoint folder that can restore the full training state.

    :param path: Checkpoint directory.
    :type path: str
    :return: The step, or ``None`` when trainer state, optimizer, scheduler or weights are missing
        or the trainer state does not parse (a folder still being written or a partial download).
    :rtype: Optional[int]
    """
    if not all(os.path.isfile(os.path.join(path, f)) for f in REQUIRED_CHECKPOINT_FILES):
        return None
    if not any(os.path.isfile(os.path.join(path, f)) for f in WEIGHT_FILES):
        return None
    with open(os.path.join(path, "trainer_state.json"), encoding="utf-8") as fh:
        text = fh.read()
    if not text.strip().endswith("}"):
        return None
    step = json.loads(text).get("global_step")
    return int(step) if isinstance(step, int) else None


def local_checkpoints(output_dir: str) -> List[str]:
    """The trainer's ``checkpoint-*`` folders under an output directory.

    :param output_dir: ``TrainingArguments.output_dir``.
    :type output_dir: str
    :return: Sorted directory paths (possibly empty).
    :rtype: List[str]
    """
    return sorted(d for d in glob.glob(os.path.join(output_dir, "checkpoint-*")) if os.path.isdir(d))


def newest_complete_checkpoint(paths: Sequence[str]) -> Tuple[Optional[str], int]:
    """The complete checkpoint with the highest ``global_step``.

    :param paths: Candidate checkpoint directories (missing or incomplete ones are skipped).
    :type paths: Sequence[str]
    :return: ``(path, step)``; ``(None, 0)`` when no candidate is complete.
    :rtype: Tuple[Optional[str], int]
    """
    best: Tuple[Optional[str], int] = (None, 0)
    for path in paths:
        if not os.path.isdir(path):
            continue
        step = checkpoint_step(path)
        if step is not None and (best[0] is None or step > best[1]):
            best = (path, step)
    return best


def resume_problems(global_step: int, expected_step: int, scheduler_step: Optional[int], lr: float,
                    expected_lr: float, n_batches: int, expected_batches: int, state_max_steps: int,
                    max_steps: int, optimizer_steps: Optional[Sequence[int]], rel_tol: float = 1e-6) -> List[str]:
    """Check the state a trainer is about to train from against the planned schedule.

    :param global_step: ``trainer.state.global_step`` after the checkpoint was loaded.
    :type global_step: int
    :param expected_step: Step of the checkpoint the session resumes from (0 = fresh).
    :type expected_step: int
    :param scheduler_step: ``lr_scheduler.last_epoch`` (``None`` when not exposed).
    :type scheduler_step: Optional[int]
    :param lr: Learning rate the scheduler reports now.
    :type lr: float
    :param expected_lr: :func:`cosine_warmup_lr` at ``expected_step``.
    :type expected_lr: float
    :param n_batches: Length of the prepared train dataloader.
    :type n_batches: int
    :param expected_batches: Batches left in the fixed row order.
    :type expected_batches: int
    :param state_max_steps: ``trainer.state.max_steps``.
    :type state_max_steps: int
    :param max_steps: Planned ``max_steps``.
    :type max_steps: int
    :param optimizer_steps: Per-parameter step counters of the optimizer (``None`` when the
        optimizer does not expose them, which is not treated as a problem).
    :type optimizer_steps: Optional[Sequence[int]]
    :param rel_tol: Relative tolerance on the learning rate.
    :type rel_tol: float
    :return: One line per mismatch; empty when the state is exactly the planned one.
    :rtype: List[str]
    """
    problems: List[str] = []
    if global_step != expected_step:
        problems.append(f"global_step is {global_step}, the checkpoint is step {expected_step}")
    if scheduler_step is not None and scheduler_step != expected_step:
        problems.append(f"the LR scheduler is at step {scheduler_step}, expected {expected_step} (schedule not restored)")
    if abs(lr - expected_lr) > rel_tol * max(abs(expected_lr), 1e-12):
        problems.append(f"learning rate is {lr:.6e}, the schedule gives {expected_lr:.6e} at step {expected_step}")
    if n_batches != expected_batches:
        problems.append(f"the train dataloader has {n_batches} batches, the fixed order has {expected_batches} left")
    if state_max_steps != max_steps:
        problems.append(f"the trainer will stop at step {state_max_steps}, the plan says {max_steps}")
    if optimizer_steps is not None:
        top = max(optimizer_steps) if optimizer_steps else 0
        if expected_step > 0 and not optimizer_steps:
            problems.append("the optimizer has no state: its moments were not restored from the checkpoint")
        elif top != expected_step:
            problems.append(f"the optimizer has taken {top} steps, expected {expected_step} (optimizer state not restored)")
    return problems


def may_retry(attempt: int, max_retries: int, failed_step: int, last_failed_step: Optional[int],
              checkpoint: Optional[int]) -> bool:
    """Whether a failed ``train()`` call may be re-entered inside the same session.

    :param attempt: Retries already used in this session.
    :type attempt: int
    :param max_retries: Retry budget.
    :type max_retries: int
    :param failed_step: ``global_step`` when the failure happened.
    :type failed_step: int
    :param last_failed_step: ``global_step`` of the previous failure in this session, if any.
    :type last_failed_step: Optional[int]
    :param checkpoint: Step of the newest complete checkpoint, or ``None`` when there is none.
    :type checkpoint: Optional[int]
    :return: True when retries remain, a complete checkpoint exists to restart from (without one
        the weights in memory are no longer the planned ones), and the failure is not a repeat at
        the same step (a row or state that fails deterministically needs a person).
    :rtype: bool
    """
    return attempt < max_retries and checkpoint is not None and failed_step != last_failed_step


# ---- notebook-only glue (needs torch / transformers / the Hub) ----
PLAN_FILE = "session_plan.json"


class ResumeGateError(AssertionError):
    """The state about to be trained from is not the planned schedule. Never retried."""


class OrderedView(_TorchDataset):
    """A map-style view of ``base`` in a fixed index order (the rows a session still has to consume)."""

    def __init__(self, base, indices):
        self.base, self.indices = base, list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return self.base[self.indices[i]]

    def column(self, name):
        col = self.base.column(name)
        return [col[j] for j in self.indices]


class _RecordingView(_TorchDataset):
    """Records the positions a dataloader fetches (order gate)."""

    def __init__(self, ds):
        self.ds, self.seen = ds, []

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        self.seen.append(int(i))
        return self.ds[i]


class TimeBudgetTrainerCallback(TrainerCallback):
    """``TimeBudgetCallback`` as a real ``TrainerCallback`` (the core stays framework-free for tests)."""

    def __init__(self, budget_s: float):
        super().__init__()
        self.core = TimeBudgetCallback(budget_s)

    def on_train_begin(self, args, state, control, **kwargs):
        return self.core.on_train_begin(args, state, control, **kwargs)

    def on_step_end(self, args, state, control, **kwargs):
        return self.core.on_step_end(args, state, control, **kwargs)


def _optimizer_steps(optimizer) -> Optional[List[int]]:
    """Per-parameter step counters of an optimizer (AdamW and the 8-bit AdamW both keep ``state[p]["step"]``).

    :param optimizer: The trainer's optimizer (possibly wrapped by Accelerate).
    :return: Step counters, or ``None`` when the optimizer does not expose them.
    """
    state = getattr(getattr(optimizer, "optimizer", optimizer), "state", None)
    if state is None:
        return None
    steps = []
    for entry in state.values():
        if isinstance(entry, dict) and "step" in entry:
            value = entry["step"]
            steps.append(int(value.item()) if hasattr(value, "item") else int(value))
    return steps


class ResumeGateTrainerCallback(TrainerCallback):
    """Before the first step of every ``train()`` call: the restored state must be the planned schedule.

    Runs in ``on_train_begin``, i.e. after the trainer loaded weights, optimizer and scheduler from the
    checkpoint and built its train dataloader. Raises :class:`ResumeGateError` on any mismatch.
    """

    def __init__(self):
        super().__init__()
        self.expected_step, self.expected_rows, self.passed = 0, 0, []

    def expect(self, step: int, rows: int) -> None:
        """Set what the next ``train()`` call must start from.

        :param step: Step of the checkpoint (0 = fresh).
        :param rows: Rows left in the fixed order.
        """
        self.expected_step, self.expected_rows = int(step), int(rows)

    def on_train_begin(self, args, state, control, optimizer=None, lr_scheduler=None, train_dataloader=None, **kwargs):
        kind = str(getattr(args.lr_scheduler_type, "value", args.lr_scheduler_type))
        if kind != "cosine":
            raise ResumeGateError(f"the resume gate knows the cosine schedule, the trainer uses {kind!r}")
        inner = getattr(optimizer, "optimizer", optimizer)
        lr = float(inner.param_groups[0]["lr"])
        expected_lr = cosine_warmup_lr(self.expected_step, args.learning_rate, args.get_warmup_steps(args.max_steps), args.max_steps)
        counters = _optimizer_steps(optimizer)
        problems = resume_problems(
            global_step=int(state.global_step), expected_step=self.expected_step,
            scheduler_step=getattr(lr_scheduler, "last_epoch", None), lr=lr, expected_lr=expected_lr,
            n_batches=len(train_dataloader), expected_batches=-(-self.expected_rows // args.per_device_train_batch_size),
            state_max_steps=int(state.max_steps), max_steps=int(args.max_steps), optimizer_steps=counters)
        if problems:
            raise ResumeGateError("resume gate FAILED before the first step:\n  - " + "\n  - ".join(problems))
        self.passed.append(self.expected_step)
        seen = "not exposed" if counters is None else (f"{len(counters)} tensors at step {max(counters)}" if counters else "empty (fresh)")
        print(f"resume gate OK: step {state.global_step} of {args.max_steps}, lr {lr:.6e} (schedule {expected_lr:.6e}), "
              f"scheduler at {getattr(lr_scheduler, 'last_epoch', '?')}, optimizer state {seen}, {self.expected_rows} rows left", flush=True)
        return control


def order_gate(trainer, view, k: int = 3) -> None:
    """The PREPARED train dataloader must hand out the view's first rows, in order.

    :param trainer: The trainer (its sampler override is already installed).
    :param view: The ordered view it is about to train on.
    :param k: Rows to check.
    :raises ResumeGateError: When the dataloader fetches anything but positions ``0..k-1``.
    """
    rec = _RecordingView(view)
    saved_ds, saved_workers = trainer.train_dataset, trainer.args.dataloader_num_workers
    trainer.train_dataset, trainer.args.dataloader_num_workers = rec, 0   # in-process fetch: positions visible, nothing prefetched
    it = iter(trainer.get_train_dataloader())
    n = min(k, len(view))
    while len(rec.seen) < n:
        next(it)
    trainer.train_dataset, trainer.args.dataloader_num_workers = saved_ds, saved_workers
    del it
    if rec.seen[:n] != list(range(n)):
        raise ResumeGateError(f"order gate FAILED: the dataloader fetched view positions {rec.seen[:n]}, expected {list(range(n))} "
                              "(the sequential sampler is not in effect)")
    print(f"order gate OK: the prepared dataloader starts with rows {[view.indices[i] for i in range(n)]} of the fixed order")


def train_with_exact_resume(trainer, base_ds, order, resume_dir, gate, max_retries: int = 2) -> int:
    """Train over the fixed row order from ``resume_dir`` (or step 0); re-enter after an in-session failure.

    Every attempt starts from a complete checkpoint: weights, optimizer and scheduler are reloaded by the
    trainer, the dataset is the tail of ``order`` that checkpoint has not consumed, the sampler is
    sequential, and both gates run before the first step. A failure is retried at most ``max_retries``
    times, never without a checkpoint and never twice at the same step.

    :param trainer: The trainer (``max_steps`` set; callbacks ``gate`` and the time budget already added).
    :param base_ds: The full train dataset (map-style).
    :param order: The materialised row order of the whole run.
    :param resume_dir: Checkpoint to start from, or ``None`` for a fresh run.
    :param gate: The :class:`ResumeGateTrainerCallback` attached to ``trainer``.
    :param max_retries: In-session retry budget.
    :return: ``global_step`` when training returned.
    """
    batch, accum = trainer.args.per_device_train_batch_size, trainer.args.gradient_accumulation_steps
    attempt, last_failed_step = 0, None
    while True:
        step = 0
        if resume_dir is not None:
            step = checkpoint_step(resume_dir)
            if step is None:
                raise ResumeGateError(f"{resume_dir} is not a complete checkpoint (trainer state, optimizer, scheduler and weights are all needed)")
        remaining = remaining_rows(order, step, batch, accum)
        if step >= trainer.args.max_steps or not remaining:
            trainer._load_from_checkpoint(resume_dir)
            print(f"run already complete at step {step}: weights loaded from {resume_dir}, nothing left to train")
            return step
        view = OrderedView(base_ds, remaining)
        trainer.train_dataset = view
        trainer.args.ignore_data_skip = True     # the skip is done by construction: the view holds only unseen rows
        trainer._get_train_sampler = lambda *a, **k: SequentialSampler(trainer.train_dataset)   # the order is baked into the view
        order_gate(trainer, view)
        gate.expect(step, len(view))
        print(f"train() attempt {attempt + 1}: from step {step}, {len(view)} rows left in the fixed order", flush=True)
        failure = None
        try:
            trainer.train(resume_from_checkpoint=resume_dir)
        except Exception as error:   # the one deliberate catch-all: a long run re-enters from its last checkpoint
            if isinstance(error, AssertionError):
                raise                # gate failures are never retried
            failure = error
        if failure is None:
            return int(trainer.state.global_step)
        failed_step = int(trainer.state.global_step)
        candidates = local_checkpoints(trainer.args.output_dir) + ([resume_dir] if resume_dir else [])
        next_dir, next_step = newest_complete_checkpoint(candidates)
        if not may_retry(attempt, max_retries, failed_step, last_failed_step, next_step if next_dir else None):
            raise failure
        print(f"⚠️ train() failed at step {failed_step}: {type(failure).__name__}: {str(failure)[:300]}\n"
              f"   in-session retry {attempt + 1}/{max_retries} from {next_dir} (step {next_step})", flush=True)
        attempt, last_failed_step, resume_dir, failure = attempt + 1, failed_step, next_dir, None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def discover_resume(ckpt_repo: str, out_dir: str, resume_root: str):
    """Newest complete checkpoint among the Hub's ``last-checkpoint`` and this machine's ``checkpoint-*`` folders.

    The Hub copy is downloaded to ``resume_root``, which must lie OUTSIDE ``out_dir``: the trainer's model
    push uploads everything in ``output_dir`` except ``checkpoint-*``, so a ``last-checkpoint`` folder
    inside it is re-uploaded at every save and rolls the Hub's resume point back to the session's start.
    Only "the repository does not exist" means a fresh start; any other Hub error propagates.

    :param ckpt_repo: Checkpoint repository id.
    :param out_dir: The trainer's ``output_dir``.
    :param resume_root: Download directory for the Hub checkpoint.
    :return: ``(checkpoint_dir or None, step)``.
    """
    from huggingface_hub import list_repo_files, snapshot_download
    from huggingface_hub.utils import RepositoryNotFoundError
    out_abs, root_abs = os.path.abspath(out_dir), os.path.abspath(resume_root)
    if root_abs == out_abs or root_abs.startswith(out_abs + os.sep):
        raise ResumeGateError(f"resume_root {resume_root} must not be inside output_dir {out_dir}")
    candidates = local_checkpoints(out_dir)
    try:
        files = list_repo_files(ckpt_repo)
    except RepositoryNotFoundError:
        files = []
        print(f"{ckpt_repo} does not exist yet")
    on_hub = any(f.startswith("last-checkpoint/") for f in files)
    if on_hub:
        snapshot_download(ckpt_repo, allow_patterns="last-checkpoint/*", local_dir=resume_root)
        candidates.append(os.path.join(resume_root, "last-checkpoint"))
    path, step = newest_complete_checkpoint(candidates)
    if on_hub and path is None:
        raise ResumeGateError(f"{ckpt_repo} has a last-checkpoint but no complete copy could be assembled (optimizer / scheduler / weights missing)")
    return path, step


def lock_session_plan(ckpt_repo: str, plan: Dict[str, object], order: Sequence[int], has_checkpoint: bool) -> List[int]:
    """Store the schedule and row order on the first launch; afterwards require the same schedule.

    :param ckpt_repo: Checkpoint repository id (exists: the trainer creates it).
    :param plan: This session's :func:`build_session_plan`.
    :param order: The row order materialised in this session.
    :param has_checkpoint: Whether a checkpoint was found to resume from.
    :return: The order to train on (the stored one from the second session on).
    :raises ResumeGateError: On any schedule difference, or a checkpoint without a stored plan.
    """
    from huggingface_hub import HfApi, hf_hub_download
    from huggingface_hub.utils import EntryNotFoundError
    try:
        with open(hf_hub_download(ckpt_repo, PLAN_FILE, force_download=True), encoding="utf-8") as fh:
            stored = json.load(fh)
    except EntryNotFoundError:
        stored = None
    if stored is None:
        if has_checkpoint:
            raise ResumeGateError(f"{ckpt_repo} has a checkpoint but no {PLAN_FILE}: it was not started by this notebook, refusing to continue it")
        payload = dict(plan, order=[int(i) for i in order], created_at=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
        HfApi().upload_file(path_or_fileobj=json.dumps(payload).encode("utf-8"), path_in_repo=PLAN_FILE, repo_id=ckpt_repo,
                            commit_message="Session plan: schedule and fixed row order (exact resume)")
        print(f"session plan stored in {ckpt_repo}/{PLAN_FILE} ({len(order)} ordered rows, sha {plan['order_sha256'][:12]})")
        return [int(i) for i in order]
    problems = diff_session_plan(stored, plan)
    if problems:
        raise ResumeGateError("this session's settings differ from the plan the run was started with:\n  - " + "\n  - ".join(problems))
    stored_order = [int(i) for i in stored["order"]]
    if order_fingerprint(stored_order) != stored["order_sha256"] or len(stored_order) != stored["order_len"]:
        raise ResumeGateError(f"the row order stored in {PLAN_FILE} does not match its own fingerprint")
    if stored["order_sha256"] != plan["order_sha256"]:
        print("⚠️ this session regenerated a different permutation (library change); training on the STORED order")
    print(f"session plan verified against {ckpt_repo}/{PLAN_FILE} (started {stored.get('created_at', '?')})")
    return stored_order


def hub_checkpoint_step(ckpt_repo: str) -> Optional[int]:
    """``global_step`` of the Hub's ``last-checkpoint`` right now.

    :param ckpt_repo: Checkpoint repository id.
    :return: The step, or ``None`` when there is no ``last-checkpoint``.
    """
    from huggingface_hub import hf_hub_download
    from huggingface_hub.utils import EntryNotFoundError
    try:
        path = hf_hub_download(ckpt_repo, "last-checkpoint/trainer_state.json", force_download=True)
    except EntryNotFoundError:
        return None
    with open(path, encoding="utf-8") as fh:
        return int(json.load(fh)["global_step"])


def ensure_hub_checkpoint(ckpt_repo: str, out_dir: str, expected_step: int) -> None:
    """The session may only end once the Hub's resume point is the step training stopped on.

    :param ckpt_repo: Checkpoint repository id.
    :param out_dir: The trainer's ``output_dir``.
    :param expected_step: ``global_step`` when ``train()`` returned.
    :raises ResumeGateError: When the checkpoint cannot be confirmed on the Hub.
    """
    step = hub_checkpoint_step(ckpt_repo)
    if step != expected_step:
        from huggingface_hub import upload_folder
        path, local_step = newest_complete_checkpoint(local_checkpoints(out_dir))
        if path is None or local_step != expected_step:
            raise ResumeGateError(f"the Hub's last-checkpoint is step {step}, training stopped at {expected_step}, "
                                  f"and no complete local checkpoint of that step exists (newest: {path} at {local_step})")
        print(f"⚠️ the Hub's last-checkpoint is step {step}, training stopped at {expected_step}: pushing {path} now", flush=True)
        upload_folder(repo_id=ckpt_repo, folder_path=path, path_in_repo="last-checkpoint",
                      commit_message=f"Training in progress, step {expected_step}, checkpoint (verified re-push)")
        step = hub_checkpoint_step(ckpt_repo)
    if step != expected_step:
        raise ResumeGateError(f"the Hub's last-checkpoint is step {step} after the push, expected {expected_step}")
    print(f"Hub resume point verified: last-checkpoint is step {step}")


MAX_STEPS = LADDER_STEPS if STAGE != "train" else len(train_ds) // ROWS_PER_STEP   # t27: one pass over the pinned mixture in its file order (a remainder of < 16 rows is not trained); a rung: 15
TIME_BUDGET_H = 7.5   # Colab: one session (runtimes end at ~8 h); the run resumes exactly
MAX_INSESSION_RETRIES = 2
ORDER_SEED = 3407
print(f"session helpers ready: MAX_STEPS={MAX_STEPS}, time budget {TIME_BUDGET_H} h, in-session retries {MAX_INSESSION_RETRIES}")


In [ ]:
# Cell 8 — text27b training (t27a / t27b, or a ladder rung): the pinned mixture in file order, exact resume, G7 + G8 + G10
import inspect
import wandb
from huggingface_hub import HfApi, hf_hub_download
from torch.utils.data import Dataset as _TDataset
from trl import SFTConfig, SFTTrainer
from unsloth import is_bf16_supported
from transformers import TrainerCallback

def family_quota(counts: Dict[str, int], k: int = 32) -> Dict[str, int]:
    """Rows to decode per family in G7.

    :param counts: Train rows per family.
    :param k: Rows per family (fewer when a family has fewer).
    :returns: ``family -> min(k, rows)``.
    """
    return {family: min(k, int(n)) for family, n in counts.items()}


def prepared_row_problems(ids: Sequence[int], labels: Sequence[int], attention: Sequence[int], expected_ids: Sequence[int],
                          n_prompt: int, prompt: str, completion: str, decode: Callable[[Sequence[int]], str],
                          think_ids: Sequence[int] = ()) -> List[str]:
    """G7: one row of a prepared (collated, right-padded) batch against the row tokenized in cell 4.

    :param ids: The batch row's input ids (padded).
    :param labels: Its labels (padded with -100).
    :param attention: Its attention mask.
    :param expected_ids: The row's ids as tokenized in cell 4.
    :param n_prompt: The row's prompt token count.
    :param prompt: The row's prompt text.
    :param completion: The row's answer + ``<|im_end|>\\n``.
    :param decode: Token ids -> text, lossless.
    :param think_ids: Ids of the think tags.
    :returns: One line per problem; empty when the prepared row is the planned one.
    """
    attention = list(attention)
    n = sum(1 for a in attention if a)
    if attention != [1] * n + [0] * (len(attention) - n):
        return ["the attention mask is not a right-padded prefix"]
    ids, labels = list(ids)[:n], list(labels)[:n]
    if ids != list(expected_ids):
        return [f"the prepared ids ({n} tokens) are not the row's cell-4 tokens ({len(expected_ids)})"]
    return supervised_problems(ids, labels, n_prompt, prompt, completion, decode, think_ids)


def milestone_steps(max_steps: int, save_steps: int, push_every: int, fractions: Sequence[float] = ()) -> List[int]:
    """Steps whose checkpoint is pushed and confirmed on the Hub before training goes on.

    :param max_steps: Steps of the run (the last is always a milestone: the trainer saves at the end of training).
    :param save_steps: Save interval (every milestone but the last is a save step).
    :param push_every: Milestone interval (a multiple of ``save_steps``).
    :param fractions: Further milestones at these fractions of the run, rounded to the nearest save step.
    :returns: Sorted milestone steps.
    :raises ValueError: On a non-positive size or a push interval that is not a multiple of the save interval.
    """
    if max_steps <= 0 or save_steps <= 0 or push_every <= 0:
        raise ValueError("max_steps, save_steps and push_every must be positive")
    if push_every % save_steps:
        raise ValueError(f"push_every {push_every} is not a multiple of save_steps {save_steps}")
    steps = set(range(push_every, max_steps + 1, push_every))
    for f in fractions:
        if not 0 < f <= 1:
            raise ValueError(f"milestone fraction {f} is not in (0, 1]")
        steps.add(min(max(int(round(f * max_steps / save_steps)) * save_steps, save_steps), max_steps))
    steps.add(max_steps)
    return sorted(steps)


def loss_alarm(value: float, reference: Optional[float], factor: float = 2.0) -> bool:
    """Whether a loss crossed ``factor`` x its step-0 reference (the gate-the-train-path rule).

    :param value: The logged loss.
    :param reference: The step-0 dev loss it is compared with (None or <= 0: no alarm).
    :param factor: Alarm level.
    :returns: True when ``value > factor * reference``.
    """
    return reference is not None and reference > 0 and value > factor * reference


RUNS = {
    "t27a": dict(run_name="genizah_t27a", ckpt_repo="isaacmg/qwen3.8-27b-genizah-t27a-ckpt", merged_repo="isaacmg/qwen3.8-27b-genizah-t27a-merged",
                 push_every=100, milestone_fractions=()),
    "t27b": dict(run_name="genizah_t27b", ckpt_repo="isaacmg/qwen3.8-27b-genizah-t27b-ckpt", merged_repo="isaacmg/qwen3.8-27b-genizah-t27b-merged",
                 push_every=250, milestone_fractions=(0.5,)),
}
RUN = RUNS[ARM]
if STAGE == "train":
    RUN_NAME, CKPT_REPO, MERGED_REPO = RUN["run_name"], RUN["ckpt_repo"], RUN["merged_repo"]   # NEW repos, private
    SAVE_STEPS, EVAL_STEPS, PUSH_EVERY, MILESTONE_FRACTIONS = 50, 100, RUN["push_every"], RUN["milestone_fractions"]
else:   # a ladder rung: scratch repos (private; delete them after the ladder), saves every KILL_AT steps
    _base = "qwen3.5-2b" if SMOKE else "qwen3.8-27b"
    RUN_NAME = f"genizah_t27_{STAGE}_{SCRATCH_TAG}"
    CKPT_REPO = f"isaacmg/{_base}-genizah-t27-{STAGE}-{SCRATCH_TAG}-ckpt"
    MERGED_REPO = f"isaacmg/{_base}-genizah-t27-{STAGE}-{SCRATCH_TAG}-merged"   # rung 1 pushes its merge test here
    SAVE_STEPS, EVAL_STEPS, PUSH_EVERY, MILESTONE_FRACTIONS = KILL_AT, KILL_AT, KILL_AT, ()
OUT_DIR = f"outputs_{RUN_NAME}"
wandb.login(key=userdata.get("WANDB_API_KEY"))

def make_sft_config(**kw):
    params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in kw and "max_seq_length" not in params:
        kw["max_length"] = kw.pop("max_seq_length")
    dropped = {k: kw.pop(k) for k in list(kw) if k not in params}
    if dropped:
        print(f"⚠️ dropped unsupported SFTConfig kwargs: {sorted(dropped)}")
    return SFTConfig(**kw)

def make_trainer(**kw):
    try:
        return SFTTrainer(**kw)
    except TypeError as e:
        if "tokenizer" in kw and ("tokenizer" in str(e) or "processing_class" in str(e)):
            kw["processing_class"] = kw.pop("tokenizer")
            return SFTTrainer(**kw)
        raise

# ---- v22b: where to resume from (helpers cell) ----
# The Hub's last-checkpoint is downloaded OUTSIDE output_dir: inside it, every model push re-uploads the stale folder
# and rolls the Hub's resume point back to this session's start (seen on the v19c / v20a / v21b resumed sessions).
# Only "the repository does not exist" is a fresh start; any other Hub error stops the notebook here.
RESUME_ROOT = "/content/t27_resume"
RESUME_ROOT = os.path.join(RESUME_ROOT, RUN_NAME)   # one folder per run, outside OUT_DIR
resume_dir, _resume_step = discover_resume(CKPT_REPO, OUT_DIR, RESUME_ROOT)
print(f"resume point: {resume_dir} (step {_resume_step})" if resume_dir else f"no checkpoint on the Hub or on this machine: fresh start of {RUN_NAME}")
assert STAGE == "train" or resume_dir is None, f"a ladder rung starts from scratch, but {CKPT_REPO} has step {_resume_step}: re-run cell 1 (a new T27_SCRATCH_TAG)"
# transformers 5 may fold the warm-up ratio into warmup_steps (a float below 1); the schedule check below holds either way
_WARMUP = {"warmup_ratio": 0.03} if "warmup_ratio" in inspect.signature(SFTConfig.__init__).parameters else {"warmup_steps": 0.03}


def sft_args(out_dir):
    """The run's SFTConfig, in one place: G10's resumed trainer is built exactly like the first."""
    return make_sft_config(
        per_device_train_batch_size=PER_DEVICE_BATCH, gradient_accumulation_steps=GRAD_ACCUM,   # 16 rows a step
        dataloader_num_workers=2, dataloader_pin_memory=True,
        max_steps=MAX_STEPS,              # one pass over the mixture; sessions end on the time budget and resume exactly
        learning_rate=1e-4, lr_scheduler_type="cosine", weight_decay=0.01, **_WARMUP,   # the resume gate knows cosine only
        logging_steps=1 if STAGE != "train" else 10,
        eval_strategy="steps", eval_steps=EVAL_STEPS, per_device_eval_batch_size=1,
        save_steps=SAVE_STEPS, save_total_limit=2,   # a termination costs at most SAVE_STEPS steps (the Hub follows every save)
        push_to_hub=True, hub_model_id=CKPT_REPO,
        hub_strategy="checkpoint", hub_private_repo=True,   # private: dataset, checkpoints and W&B stay private
        hub_always_push=True,             # every save is pushed, also the time-budget stop that lands right after a regular save
        optim='adamw_8bit', seed=3407, data_seed=3407, output_dir=out_dir,
        report_to="wandb", run_name=RUN_NAME,
        bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
        remove_unused_columns=False, dataset_text_field="",
        accelerator_config={"dispatch_batches": False},   # map-style rows: Accelerate does not dispatch, the v2.1 rule stays gated
        dataset_kwargs={"skip_prepare_dataset": True},     # rows are rendered and tokenized in cell 4 (prompt masked)
        max_seq_length=MAX_SEQ,
    )


FastModel.for_training(model)
from datasets import Dataset as _HFDataset


def trl_dataset(rows: TokenizedRows) -> _HFDataset:
    """Expose pretokenized rows through TRL 1.15's supported dataset interface.

    :param rows: Tokenized rows in their original order, with response-only labels.
    :returns: A map-style Hugging Face dataset yielding the same tensors.
    """
    return _HFDataset.from_dict({
        "input_ids": [row.tolist() for row in rows.ids],
        "labels": [rows[i]["labels"].tolist() for i in range(len(rows))],
    }).with_format("torch")


_trl_train_ds = trl_dataset(train_ds)
_trl_dev_sets = {family: trl_dataset(rows) for family, rows in dev_sets.items()}
trainer = make_trainer(model=model, tokenizer=_tok, data_collator=collate, train_dataset=_trl_train_ds, eval_dataset=_trl_dev_sets,
                       args=sft_args(OUT_DIR))
# schedule check (G8): make_sft_config drops kwargs this TRL does not know — the warm-up and the cosine must have survived
assert trainer.args.get_warmup_steps(MAX_STEPS) == warmup_steps_for(MAX_STEPS, 0.03), (
    f"warm-up {trainer.args.get_warmup_steps(MAX_STEPS)} steps != planned {warmup_steps_for(MAX_STEPS, 0.03)}")
assert str(getattr(trainer.args.lr_scheduler_type, "value", trainer.args.lr_scheduler_type)) == "cosine", "not a cosine schedule"
wandb.init(project=WANDB_PROJECT, name=RUN_NAME, config={
    "arm": ARM, "stage": STAGE, "resume_step": _resume_step, "base_model": BASE_MODEL, "data_repo": DATA_REPO,
    "data_revision": DATA_REVISION, "mixture_fingerprint": MIXTURE_FINGERPRINT, "pin_status": T27_PIN_STATUS,
    "resolved_pins": RESOLVED_PINS, "chat_template_sha256": TEMPLATE_SHA256, "lora_regex": LORA_REGEX,
    "lora_counts": LORA_COUNTS, "batch": [PER_DEVICE_BATCH, GRAD_ACCUM], "max_steps": MAX_STEPS, "mixture": MIX}, allow_val_change=True)

# ---- G7: the PREPARED train dataloader (v2.1 lesson) hands out the planned rows: 32 rows per family decoded ----
assert trainer.args.accelerator_config.dispatch_batches is False, "dispatch_batches=False did not reach the trainer"

class _Recording(_TDataset):
    """Wrapper that records fetched indices so the gate can find the row behind a prepared batch."""
    def __init__(self, ds):
        self.ds, self.seen = ds, []
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        self.seen.append(int(i))
        return self.ds[i]

_rec = _Recording(train_ds)
trainer.train_dataset = _rec
_saved_workers, trainer.args.dataloader_num_workers = trainer.args.dataloader_num_workers, 0  # in-process fetch ⇒ indices visible
_dl = trainer.get_train_dataloader()
assert "Dispatcher" not in type(_dl).__name__, f"train dataloader is {type(_dl).__name__} — batches would be truncated"
_want = family_quota(Counter(train_ds.column("family")), 32)
_got, _pos, _first = Counter(), 0, None
for _b in _dl:
    if _first is None:
        _first, _first_rows = _b, list(_rec.seen[:_b["input_ids"].shape[0]])
    for _j in range(_b["input_ids"].shape[0]):
        _i = _rec.seen[_pos]
        _pos += 1
        _fam = train_ds.meta[_i]["family"]
        if _got[_fam] >= _want[_fam]:
            continue
        _prompt, _completion = train_ds.parts(_i)
        _p = prepared_row_problems(_b["input_ids"][_j].tolist(), _b["labels"][_j].tolist(), _b["attention_mask"][_j].tolist(),
                                   train_ds.ids[_i].tolist(), train_ds.n_prompt[_i], _prompt, _completion, decode, THINK_IDS)
        assert not _p, f"G7 FAILED on prepared row {_i} ({_fam}): {_p}"
        _got[_fam] += 1
    if all(_got[_f] >= _n for _f, _n in _want.items()):
        break
assert all(_got[_f] >= _n for _f, _n in _want.items()), f"G7: decoded {dict(_got)}, wanted {_want}"
print(f"G7 OK: {sum(_got.values())} prepared rows {dict(_got)} = the cell-4 tokens; prompt + empty think block masked, answer supervised")
# ---- G8: the loss on the first prepared batch equals the loss on the same rows collated directly ----
_direct = {_k: _v.to(model.device) for _k, _v in collate([train_ds[_i] for _i in _first_rows]).items()}
assert torch.equal(_direct["input_ids"], _first["input_ids"].to(model.device)), "G8: prepared batch 0 is not the recorded rows"
model.eval()
with torch.no_grad():
    _lp = model(**{_k: _v.to(model.device) for _k, _v in _first.items()}).loss.item()
    _ld = model(**_direct).loss.item()
assert abs(_lp - _ld) <= 0.05 * max(1.0, _ld), f"G8: prepared-dataloader loss {_lp:.3f} != direct-collate loss {_ld:.3f}"
print(f"G8 OK: loss prepared={_lp:.3f} direct={_ld:.3f} (rows {_first_rows}); warm-up {trainer.args.get_warmup_steps(MAX_STEPS)} steps")
trainer.train_dataset = train_ds
trainer.args.dataloader_num_workers = _saved_workers
del _dl, _b, _first, _direct, _rec
FastModel.for_training(model)

# ---- G9 (every fresh start): the step-0 dev loss per family is the 2x loss-alarm reference ----
STEP0_FILE = "step0_dev_loss.json"
if resume_dir is None:
    _m0 = trainer.evaluate()
    STEP0_DEV_LOSS = {_k[len("eval_"):-len("_loss")]: float(_v) for _k, _v in _m0.items() if _k.startswith("eval_") and _k.endswith("_loss")}
    FastModel.for_training(model)
else:
    with open(hf_hub_download(CKPT_REPO, STEP0_FILE), encoding="utf-8") as _fh:
        STEP0_DEV_LOSS = json.load(_fh)
print("step-0 dev loss per family:", {_k: round(_v, 3) for _k, _v in sorted(STEP0_DEV_LOSS.items())})

# ---- v22b: schedule lock, fixed row order, exact resume (helpers cell) — t27: the order is the mixture file's own ----
_a = trainer.args
_order = list(range(len(train_ds))) * epochs_for(MAX_STEPS, len(train_ds), _a.per_device_train_batch_size, _a.gradient_accumulation_steps)
_plan = build_session_plan(
    _order, order_description="mixture file order", n_rows=len(train_ds), max_steps=_a.max_steps,
    per_device_batch=_a.per_device_train_batch_size, grad_accum=_a.gradient_accumulation_steps,
    learning_rate=_a.learning_rate, warmup_ratio=0.03, lr_scheduler_type=str(getattr(_a.lr_scheduler_type, "value", _a.lr_scheduler_type)),
    weight_decay=_a.weight_decay, optim=str(getattr(_a.optim, "value", _a.optim)), seed=_a.seed,
    dataset_repo=DATA_REPO, dataset_revision=DATA_REVISION, rows_sha256=MIXTURE_FINGERPRINT, base_model=BASE_MODEL,
    lora=f"r{LORA_RANK}/alpha{LORA_ALPHA}/{LORA_REGEX}", max_seq=MAX_SEQ, render="enable_thinking=False")
_order = lock_session_plan(CKPT_REPO, _plan, _order, has_checkpoint=resume_dir is not None)   # first launch stores it; later sessions must match it
if resume_dir is None:
    HfApi().upload_file(path_or_fileobj=json.dumps(STEP0_DEV_LOSS).encode("utf-8"), path_in_repo=STEP0_FILE, repo_id=CKPT_REPO,
                        commit_message="Step-0 dev loss per family (2x loss-alarm reference)")
print(describe_session(len(train_ds), MAX_STEPS, _a.per_device_train_batch_size, _a.gradient_accumulation_steps, _resume_step, _a.seed))

# ---- milestones: pushed AND confirmed on the Hub before training goes on (the Studio evals; t27b's 50 % / 100 % comparison) ----
MILESTONES = milestone_steps(MAX_STEPS, SAVE_STEPS, PUSH_EVERY, MILESTONE_FRACTIONS)
print(f"Hub-confirmed milestones: {MILESTONES}")


class MilestonePushCallback(TrainerCallback):
    """At every milestone save (and at a train() start on a milestone), wait for the trainer's Hub pushes and confirm
    that the Hub's last-checkpoint is that step."""

    def __init__(self, trainer, ckpt_repo, out_dir, steps):
        super().__init__()
        self.trainer, self.ckpt_repo, self.out_dir, self.steps, self.confirmed = trainer, ckpt_repo, out_dir, {int(s) for s in steps}, []

    def confirm(self, step):
        if step not in self.steps or step in self.confirmed:
            return
        job = getattr(self.trainer, "push_in_progress", None)
        if job is not None:
            job.wait_until_done()          # the pushes run one after another: this step's is the last one queued
        ensure_hub_checkpoint(self.ckpt_repo, self.out_dir, step)
        self.confirmed.append(step)
        print(f"milestone step {step}: checkpoint confirmed on the Hub ({self.ckpt_repo})", flush=True)

    def on_train_begin(self, args, state, control, **kwargs):
        self.confirm(int(state.global_step))   # an in-session retry or a resume from a milestone confirms it first
        return control

    def on_save(self, args, state, control, **kwargs):
        self.confirm(int(state.global_step))
        return control


class LossAlarmCallback(TrainerCallback):
    """Alarm (print + W&B alert, training goes on) when a loss crosses 2x its step-0 dev reference."""

    def __init__(self, reference, families, factor):
        super().__init__()
        self.reference, self.factor, self.fired = dict(reference), float(factor), set()
        trained = [self.reference[f] for f in families if f in self.reference]
        self.train_reference = sum(trained) / len(trained) if trained else None

    def _alarm(self, key, text):
        if key in self.fired:
            return
        self.fired.add(key)
        print(f"🚨 LOSS ALARM: {text}", flush=True)
        if wandb.run is not None:
            wandb.alert(title=f"{RUN_NAME}: loss alarm", text=text)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        for key, value in (metrics or {}).items():
            family = key[len("eval_"):-len("_loss")] if key.startswith("eval_") and key.endswith("_loss") else None
            if family and loss_alarm(float(value), self.reference.get(family), self.factor):
                self._alarm(("eval", family), f"step {state.global_step}: dev {family} loss {value:.3f} > {self.factor} x step-0 {self.reference[family]:.3f}")
        return control

    def on_log(self, args, state, control, logs=None, **kwargs):
        value = (logs or {}).get("loss")
        if value is not None and loss_alarm(float(value), self.train_reference, self.factor):
            self._alarm(("train", state.global_step // 100), f"step {state.global_step}: train loss {value:.3f} > {self.factor} x step-0 dev {self.train_reference:.3f}")
        return control


class KillAtStepCallback(TrainerCallback):
    """Ladder rungs (G10): end train() at ``step`` with a save, as a killed session would have left it."""

    def __init__(self, step):
        super().__init__()
        self.step = int(step)

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step == self.step:
            control.should_save = True
            control.should_training_stop = True
        return control


def add_callbacks(trainer, kill_at=None):
    """The session's callbacks: resume gate, time budget, milestone pushes, loss alarm (and a rung's kill)."""
    gate = ResumeGateTrainerCallback()
    trainer.add_callback(gate)                                                      # step / LR / optimizer state / rows left, before the first step
    trainer.add_callback(TimeBudgetTrainerCallback(budget_s=TIME_BUDGET_H * 3600))   # clean save + push before the session's end
    trainer.add_callback(MilestonePushCallback(trainer, CKPT_REPO, trainer.args.output_dir, MILESTONES))
    trainer.add_callback(LossAlarmCallback(STEP0_DEV_LOSS, ARM_FAMILIES[ARM], 2.0))
    if kill_at is not None:
        trainer.add_callback(KillAtStepCallback(kill_at))
    return gate


_gate = add_callbacks(trainer, KILL_AT if STAGE != "train" else None)
_final_step = train_with_exact_resume(trainer, train_ds, _order, resume_dir, _gate, max_retries=MAX_INSESSION_RETRIES)
if STAGE != "train":
    # ---- G10: the session was "killed" at step KILL_AT; resume exactly from the Hub's copy in a fresh trainer ----
    assert _final_step == KILL_AT, f"G10: the rung stopped at step {_final_step}, expected the kill at {KILL_AT}"
    ensure_hub_checkpoint(CKPT_REPO, OUT_DIR, KILL_AT)
    _resumed_out, _resumed_root = OUT_DIR + "_resumed", RESUME_ROOT + "_resumed"
    _hub_dir, _hub_step = discover_resume(CKPT_REPO, _resumed_out, _resumed_root)   # a new output dir: only the Hub's copy is a candidate
    assert _hub_step == KILL_AT and _hub_dir == os.path.join(_resumed_root, "last-checkpoint"), f"G10: resume point {_hub_dir} (step {_hub_step})"
    _sha = lambda _path: hashlib.sha256(Path(_path).read_bytes()).hexdigest()
    assert _sha(os.path.join(OUT_DIR, f"checkpoint-{KILL_AT}", "adapter_model.safetensors")) == _sha(os.path.join(_hub_dir, "adapter_model.safetensors")), \
        "G10: the Hub's adapter differs from the one this session saved"
    del trainer                                                                      # a new session: nothing of the first trainer survives
    gc.collect()
    torch.cuda.empty_cache()
    trainer = make_trainer(model=model, tokenizer=_tok, data_collator=collate, train_dataset=_trl_train_ds, eval_dataset=_trl_dev_sets,
                           args=sft_args(_resumed_out))
    _order = lock_session_plan(CKPT_REPO, _plan, _order, has_checkpoint=True)      # the stored plan must match this session's
    _gate = add_callbacks(trainer)
    _final_step = train_with_exact_resume(trainer, train_ds, _order, _hub_dir, _gate, max_retries=0)
    assert _final_step == MAX_STEPS and _gate.passed == [KILL_AT], f"G10: resumed run ended at {_final_step}, gate passes {_gate.passed}"
    OUT_DIR = _resumed_out
    print(f"G10 OK: killed at step {KILL_AT}, resumed from the Hub copy (same adapter bytes, resume gate passed), finished at step {_final_step}")
ensure_hub_checkpoint(CKPT_REPO, OUT_DIR, _final_step)                           # the session ends only once the Hub holds this step
print(f"session ended at step {_final_step} of {MAX_STEPS}" + ("" if _final_step >= MAX_STEPS else ": run the notebook again to continue from exactly here"))


In [ ]:
# Cell 9 — export: shipped-adapter check, merge to bf16 (G10 merge test on the rungs), push (private)
import shutil

def merged_dir_problems(files: Sequence[str], weight_keys: Sequence[str], config: Dict[str, Any],
                        has_chat_template: bool) -> List[str]:
    """G10 merge test on a merged (bf16) model folder.

    :param files: File names in the folder.
    :param weight_keys: Tensor names of its weights.
    :param config: Its ``config.json``.
    :param has_chat_template: Whether the chat template was saved with it.
    :returns: One line per problem; empty when the folder is a complete, adapter-free model.
    """
    problems: List[str] = []
    if "config.json" not in files:
        problems.append("no config.json")
    if not any(f.endswith(".safetensors") for f in files):
        problems.append("no safetensors weights")
    if "qwen3_5" not in str(config.get("model_type", "")):
        problems.append(f"model_type {config.get('model_type')!r} is not qwen3_5")
    lora = [k for k in weight_keys if "lora_" in k]
    if lora:
        problems.append(f"{len(lora)} LoRA tensors left in the merge, e.g. {lora[:2]}")
    if not any(".layers." in k for k in weight_keys):
        problems.append("no language-layer weights")
    if config.get("vision_config") and not any("visual" in k or "vision" in k for k in weight_keys):
        problems.append("the config has a vision tower but the merge has no vision weights")
    if not has_chat_template:
        problems.append("no chat template saved with the merge")
    return problems

MERGED_DIR = f"{RUN_NAME}-merged"
MERGED_GB = 4.5 if SMOKE else 55.6   # bf16 size of the merged model on disk
if STAGE == "train" and _final_step < MAX_STEPS:
    print(f"run stopped at step {_final_step} of {MAX_STEPS}: nothing exported — run the notebook again to continue")
else:
    # the shipped adapter: every language lora_B moved, none on vision / MTP (the G3 plan)
    _moved = [_n.replace("lora_B", "lora_A") for _n, _p in model.named_parameters() if "lora_B" in _n and _p.detach().abs().max().item() > 0]
    _counts, _problems = lora_problems(_moved, LAYER_TYPES, EXPECTED_LORA_A)
    assert not _problems, "shipped adapter: lora_B tensors that never moved or sit outside the language layers:\n  - " + "\n  - ".join(_problems)
    _free = shutil.disk_usage(".").free / 1e9
    assert _free > 1.2 * MERGED_GB + 5, f"{_free:.0f} GB free; the bf16 merge needs about {MERGED_GB} GB"
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
    _files = sorted(os.listdir(MERGED_DIR))
    with open(os.path.join(MERGED_DIR, "config.json"), encoding="utf-8") as _fh:
        _merged_cfg = json.load(_fh)
    if "model.safetensors.index.json" in _files:
        with open(os.path.join(MERGED_DIR, "model.safetensors.index.json"), encoding="utf-8") as _fh:
            _keys = list(json.load(_fh)["weight_map"])
    else:
        from safetensors import safe_open
        with safe_open(os.path.join(MERGED_DIR, "model.safetensors"), "pt") as _fh:
            _keys = list(_fh.keys())
    _has_template = "chat_template.jinja" in _files
    if not _has_template and "tokenizer_config.json" in _files:
        with open(os.path.join(MERGED_DIR, "tokenizer_config.json"), encoding="utf-8") as _fh:
            _has_template = "chat_template" in json.load(_fh)
    _problems = merged_dir_problems(_files, _keys, _merged_cfg, _has_template)
    assert not _problems, "G10 merge test FAILED:\n  - " + "\n  - ".join(_problems)
    print(f"merged to {MERGED_DIR}: {len(_keys):,} tensors, {sum(os.path.getsize(os.path.join(MERGED_DIR, _f)) for _f in _files) / 1e9:.1f} GB")
    if STAGE == "rung1":
        # G10: the merged bf16 model scores a dev row like the 4-bit model + adapter
        import transformers as _tf
        _merged = getattr(_tf, _merged_cfg["architectures"][0]).from_pretrained(MERGED_DIR, torch_dtype=torch.bfloat16, device_map="cuda")
        _b = {_k: _v.to("cuda") for _k, _v in collate([next(iter(dev_sets.values()))[0]]).items()}
        model.eval()
        with torch.no_grad():
            _la, _lm = model(**_b).loss.item(), _merged(**_b).loss.item()
        del _merged
        torch.cuda.empty_cache()
        assert abs(_lm - _la) <= 0.05 * max(1.0, _la), f"G10: merged bf16 loss {_lm:.3f} != 4-bit + adapter loss {_la:.3f}"
        print(f"G10 OK: merged bf16 loss {_lm:.3f}, 4-bit + adapter {_la:.3f}")
    if STAGE in ("rung1", "train"):
        model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", private=True)
        print("pushed", MERGED_REPO, "(private)")
    else:
        print(f"rung 2: merge test only — {MERGED_DIR} is not pushed (delete it to free the disk)")
